## imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, 
                            mean_squared_error, mean_absolute_error, confusion_matrix,
                            f1_score)
from scipy import stats
import re
import ast
import warnings
from __future__ import annotations

from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Any, Optional, Set

warnings.filterwarnings('ignore')
# Set matplotlib parameters for better looking plots
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']

## else

In [ ]:
import pandas as pd
df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv", encoding="utf-8-sig")
pred = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\PRED.csv", encoding="utf-8-sig")
# remove role_x and role_y from pred if exists
# Criminal_record,role, laboratory

cols_to_replace = ['role']
pred = pred.drop(columns=cols_to_replace)

pred = pred.merge(df[['original_case', 'role']] , on='original_case', how='left')
pred.to_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\PRED.csv",
    index=False,
    encoding="utf-8-sig"
)


In [ ]:
import pandas as pd

# Load files
features_path = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv"
gt_path = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\GT.csv"

df = pd.read_csv(features_path, encoding="utf-8-sig")
gt = pd.read_csv(gt_path, encoding="utf-8-sig")

# Keep only cases not already in GT (by case ID)
new_pool = df[~df["original_case"].isin(gt["original_case"])].copy()

# Sample 25 random cases (change random_state if you want different sample)
sample_25 = new_pool.sample(n=25, random_state=42)

# Align to GT columns only (safe append)
sample_25_for_gt = sample_25.reindex(columns=gt.columns)

# Append
gt_updated = pd.concat([gt, sample_25_for_gt], ignore_index=True)
gt_path = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\GT1.csv"

# Save back
gt_updated.to_csv(gt_path, index=False, encoding="utf-8-sig")

print(f"Added {len(sample_25_for_gt)} cases. GT now has {len(gt_updated)} rows.")


## create pred csv

In [ ]:
gt = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\GT.csv", encoding="utf-8-sig")
df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv", encoding="utf-8-sig")

# take all the rows in df that are in gt original_case and all columns in gt anf create new pred from df
df1 = df[df['original_case'].isin(gt['original_case'])].copy()
print(len(df1))
df1 = df1.reset_index(drop=True)

df1.to_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\PRED.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv", encoding="utf-8-sig")
# keep only rpws when age column is under 15 make it int
#drop all the rows that have לא ידוע in age column
df = df[df['age'] != 'לא ידוע'].copy()
df['age'] = df['age'].astype(int)
df_under_15 = df[df['age'] < 15].copy()
folder_path = r'C:\Users\user\OneDrive\טל\projects\justice-tal\bias\CSV_OUTPUT'
texts = []
# for original_case in df_under_15['original_case'] take the text from the file in folder_path with the same name and save it in a list
for original_case in df_under_15['original_case']:
    file_path = f"{folder_path}/{original_case}.csv"
    file_df = pd.read_csv(file_path, encoding="utf-8-sig")
    text = file_df['text'].tolist()
    # combine all the text in the list to one string and save it in a new list
    text = "\n".join(text)
    texts.append(text)

# add the text to the df_under_15
df_under_15['text'] = texts
print(len(df_under_15))
columns = ['original_case', 'age', 'text']
df_under_15 = df_under_15[columns]
df_under_15.to_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\under_15_cases.csv",
    index=False,
    encoding="utf-8-sig"
)


## check drug amounts error

In [ ]:
import ast
import re
import numpy as np
import pandas as pd

# =========================
# CONSTANTS
# =========================
SIGNS = ['sign_a_non_personal', 'sign_a_personal', 'sign_b']

drug_cols = [
    'lsd', 'methamphetamine', 'ayahuasca', 'cathinone', 'ketamine',
    'hashish', 'methylmethcathinone', 'cannabis_plants', 'cannabis',
    'mdma', 'cocaine', 'heroin'
]

DRUG_COLS = [
    'lsd', 'methamphetamine', 'ayahuasca', 'cathinone', 'ketamine',
    'hashish', 'methylmethcathinone', 'cannabis_plants', 'cannabis',
    'mdma', 'cocaine', 'heroin'
]

# =========================
# 1) DEDUPLICATION BY SIGN PRIORITY
# sign_b > sign_a_personal > sign_a_non_personal
# =========================
def dedupe_drug_cell(cell):
    if cell is None or (isinstance(cell, float) and np.isnan(cell)):
        return cell

    if isinstance(cell, str):
        try:
            cell = ast.literal_eval(cell)
        except Exception:
            return cell

    if not isinstance(cell, dict):
        return cell

    def canon(x):
        if x is None:
            return ""
        s = str(x)
        s = re.sub(r"\s+", "", s)
        s = s.replace("״", '"').replace("׳", "'")
        s = re.sub(r"(?<=\d),(?=\d)", ".", s)  # 0,5 -> 0.5
        return s

    non_p = cell.get('sign_a_non_personal', []) or []
    pers  = cell.get('sign_a_personal', []) or []
    b     = cell.get('sign_b', []) or []

    non_p_map = {canon(x): x for x in non_p}
    pers_map  = {canon(x): x for x in pers}
    b_map     = {canon(x): x for x in b}

    # remove duplicates from non_personal if exist in higher priority
    for k in list(non_p_map.keys()):
        if k in pers_map or k in b_map:
            non_p_map.pop(k, None)

    cell['sign_a_non_personal'] = list(non_p_map.values())
    cell['sign_a_personal'] = list(pers_map.values())
    cell['sign_b'] = list(b_map.values())

    return cell

def remove_similar_quantities(quantity_list, tolerance_percent=0.1):
    """
    Remove quantities that are very close to each other (likely duplicates with different precision)
    ומ
    Args:
        quantity_list: list of quantity strings like ['8.717-ק"ג', '8.7-ק"ג']
        tolerance_percent: if two quantities differ by less than this %, consider them duplicates
    
    Returns:
        filtered list with similar quantities removed (keeps the one with more precision)
    """
    if not quantity_list or len(quantity_list) <= 1:
        return quantity_list
    
    # Parse each quantity into (amount, unit, original_string, precision)
    parsed = []
    for q_str in quantity_list:
        if not q_str:
            continue
        
        # Clean up
        clean_str = q_str.replace('\\"', '"').replace("\\'", "'")
        clean_str = clean_str.replace('״', '"').replace('׳', "'")
        
        # Extract amount and unit
        match = re.match(r'^\[?([0-9.,]+)\s*-\s*([^\]]+)\]?$', clean_str.strip())
        if match:
            amount_str = match.group(1).replace(',', '.')
            unit = match.group(2).strip()
            try:
                amount = float(amount_str)
                # Count decimal places for precision
                precision = len(amount_str.split('.')[-1]) if '.' in amount_str else 0
                parsed.append({
                    'amount': amount,
                    'unit': unit,
                    'original': q_str,
                    'precision': precision
                })
            except ValueError:
                parsed.append({'original': q_str})  # Keep unparseable as-is
    
    if not parsed:
        return quantity_list
    
    # Group by unit
    by_unit = {}
    unparseable = []
    
    for item in parsed:
        if 'amount' in item:
            unit = item['unit']
            if unit not in by_unit:
                by_unit[unit] = []
            by_unit[unit].append(item)
        else:
            unparseable.append(item['original'])
    
    # For each unit, remove similar quantities
    result = []
    for unit, items in by_unit.items():
        if len(items) == 1:
            result.append(items[0]['original'])
            continue
        
        # Sort by amount
        items_sorted = sorted(items, key=lambda x: x['amount'])
        
        # Mark items to keep
        keep = [True] * len(items_sorted)
        
        for i in range(len(items_sorted)):
            if not keep[i]:
                continue
            
            for j in range(i + 1, len(items_sorted)):
                if not keep[j]:
                    continue
                
                amt_i = items_sorted[i]['amount']
                amt_j = items_sorted[j]['amount']
                
                # Calculate percentage difference
                avg = (amt_i + amt_j) / 2
                diff_percent = abs(amt_i - amt_j) / avg * 100
                if diff_percent <= tolerance_percent:
                    # print(f"first = {amt_i}, second = {amt_j}, list = {quantity_list}")
                    # Keep the one with more precision (more decimal places)
                    if items_sorted[i]['precision'] >= items_sorted[j]['precision']:
                        keep[j] = False
                    else:
                        keep[i] = False
                        break
        
        # Add kept items
        for i, item in enumerate(items_sorted):
            if keep[i]:
                result.append(item['original'])
    
    result.extend(unparseable)
    return result

# =========================
# 2) EXTRACT ALL QUANTITIES (AFTER DEDUPE)
# =========================
def extract_all_quantities(cell):
    if cell is None or (isinstance(cell, float) and np.isnan(cell)):
        return []

    if isinstance(cell, str):
        try:
            cell = ast.literal_eval(cell)
        except Exception:
            return []

    if not isinstance(cell, dict):
        return []

    quantities = []
    for sign in SIGNS:
        vals = cell.get(sign, [])
        if isinstance(vals, list):
            quantities.extend(vals)

    return quantities

# =========================
# 3) NORMALIZE QUANTITIES (SUM)
# =========================
def normalize_drug_quantity(quantity_str):
    if pd.isna(quantity_str) or not str(quantity_str).strip():
        return ""

    quantity_str = str(quantity_str).strip()
    total_grams = 0.0
    total_tablets = 0.0
    total_plants = 0.0
    has_valid = False

    # Clean up escaped quotes first
    quantity_str = quantity_str.replace('\\"', '"').replace("\\'", "'")
    quantity_str = quantity_str.replace('״', '"').replace('׳', "'")

    bracket_content = re.findall(r'\[(.*?)\]', quantity_str)
    all_pairs = []

    for content in bracket_content:
        parts = content.split(',')
        for part in parts:
            part = part.strip()
            match = re.match(r'^([0-9.,]+)\s*-\s*(.+)$', part)
            if match:
                all_pairs.append((match.group(1), match.group(2)))

    if not all_pairs:
        pattern = r'\[([0-9.,]+)-([^\]]+)\]'
        all_pairs = re.findall(pattern, quantity_str)
    
    # If still no pairs found, try to parse without brackets
    if not all_pairs:
        # Handle formats like: 27.5-ק"ג or [27.5-ק"ג]
        pattern = r'([0-9.,]+)\s*-\s*([^\[\],]+)'
        all_pairs = re.findall(pattern, quantity_str)

    for amount_str, unit in all_pairs:
        unit = unit.strip()
        amount_str = amount_str.replace(',', '.').strip()

        try:
            amount = float(amount_str)
            has_valid = True
        except ValueError:
            continue

        if unit in ['ק"ג', 'קילוגרם', 'קילו', 'kg', 'קג']:
            total_grams += amount * 1000
        elif unit in ['גרם', 'gram', 'grams', 'gr', 'g']:
            total_grams += amount
        elif unit in ['מ"ג', 'מיליגרם', 'mg']:
            total_grams += amount / 1000
        elif unit in ['טבליות', 'כדורים', 'יחידות', 'tablets', 'pills', 'בולים', 'כדור', 'טבליה']:
            total_tablets += amount
        elif unit in ['שתילים', 'שתיל', 'plants', 'צמחים']:
            total_plants += amount

    if not has_valid:
        return ""

    parts = []
    if total_grams > 0:
        parts.append(f"[{int(total_grams) if total_grams.is_integer() else round(total_grams, 2)}-גרם]")
    if total_tablets > 0:
        parts.append(f"[{int(total_tablets)}-טבליות]")
    if total_plants > 0:
        parts.append(f"[{int(total_plants)}-שתילים]")

    return ", ".join(parts) if parts else ""

def normalize_from_list(quantity_list):
    if quantity_list is None or not isinstance(quantity_list, list) or len(quantity_list) == 0:
        return ""
    
    # Remove similar quantities first
    filtered_list = remove_similar_quantities(quantity_list, tolerance_percent=0.01)
    
    joined = "[" + ", ".join(filtered_list) + "]"
    return normalize_drug_quantity(joined)

# =========================
# 4) EXTRACT NUMERIC VALUE
# =========================
def extract_numeric_value(normalized_str):
    if pd.isna(normalized_str) or not str(normalized_str).strip():
        return np.nan
    match = re.search(r'\[([0-9.]+)-', str(normalized_str))
    if match:
        try:
            return float(match.group(1))
        except ValueError:
            return np.nan
    return np.nan



In [ ]:
# =========================
# Configuration
# =========================
drug_cols = [
    'lsd', 'methamphetamine', 'ayahuasca', 'cathinone', 'ketamine',
    'hashish', 'methylmethcathinone', 'cannabis_plants', 'cannabis',
    'mdma', 'cocaine', 'heroin'
]

# =========================
# Load Data
# =========================
df_gt = pd.read_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\GT.csv",
    encoding="utf-8-sig"
)
df_pred = pd.read_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\PRED.csv",
    encoding="utf-8-sig"
)

# =========================
# Align DataFrames
# =========================
def align_on_case(df1_g, df2_g, key="original_case"):
    common = np.intersect1d(
        df1_g[key].dropna().unique(),
        df2_g[key].dropna().unique()
    )
    
    df1_a = (df1_g[df1_g[key].isin(common)]
             .sort_values(key)
             .reset_index(drop=True))
    
    df2_a = (df2_g[df2_g[key].isin(common)]
             .sort_values(key)
             .reset_index(drop=True))
    
    return df1_a, df2_a

df_gt, df_pred = align_on_case(df_gt, df_pred, key="original_case")
for col in drug_cols:
    # 1) remove duplicates by sign priority
    df_gt[col] = df_gt[col].apply(dedupe_drug_cell)
    df_pred[col] = df_pred[col].apply(dedupe_drug_cell)

    # 2) flatten quantities
    df_gt[col] = df_gt[col].apply(extract_all_quantities)
    df_pred[col] = df_pred[col].apply(extract_all_quantities)

    # 3) normalize & sum
    df_gt[col] = df_gt[col].apply(normalize_from_list)
    df_pred[col] = df_pred[col].apply(normalize_from_list)

    # 4) extract numeric value
    df_gt[col] = df_gt[col].apply(extract_numeric_value)
    df_pred[col] = df_pred[col].apply(extract_numeric_value)

print(f"Aligned DataFrames: {len(df_gt)} rows")
print("=" * 80)

# =========================
# Find and Display Differences
# =========================
differences = []

for idx in range(len(df_gt)):
    case_id = df_gt.loc[idx, 'original_case']
    ethnicity = df_gt.loc[idx, 'Defendants_Ethnic']
    
    for drug in DRUG_COLS:
        gt_val = pd.to_numeric(df_gt.loc[idx, drug], errors='coerce')
        pred_val = pd.to_numeric(df_pred.loc[idx, drug], errors='coerce')
        
        # Check if values are different (including NaN != number cases)
        is_different = False
        
        # Case 1: One is NaN and other is not
        if pd.isna(gt_val) != pd.isna(pred_val):
            is_different = True
        # Case 2: Both are numbers but different
        elif pd.notna(gt_val) and pd.notna(pred_val):
            if abs(gt_val - pred_val) > 0.001:  # Small tolerance for floating point
                is_different = True
        
        if is_different:
            differences.append({
                'case_id': case_id,
                'ethnicity': ethnicity,
                'drug': drug,
                'gt_quantity': gt_val if pd.notna(gt_val) else 'N/A',
                'pred_quantity': pred_val if pd.notna(pred_val) else 'N/A',
                'difference': abs(gt_val - pred_val) if pd.notna(gt_val) and pd.notna(pred_val) else 'N/A'
            })

# =========================
# Display Results
# =========================
if differences:
    df_diff = pd.DataFrame(differences)
    
    print(f"\n{'=' * 80}")
    print(f"FOUND {len(df_diff)} DIFFERENCES IN DRUG QUANTITIES")
    print(f"{'=' * 80}\n")
    
    # Sort by ethnicity and drug for easier reading
    df_diff = df_diff.sort_values(['ethnicity', 'drug', 'case_id'])
    
    # Display each difference
    for _, row in df_diff.iterrows():
        print(f"Case ID: {row['case_id']}")
        print(f"Ethnicity: {row['ethnicity']}")
        print(f"Drug: {row['drug']}")
        print(f"  GT:   {row['gt_quantity']}")
        print(f"  PRED: {row['pred_quantity']}")
        if row['difference'] != 'N/A':
            print(f"  Diff: {row['difference']:.2f}")
        print()
    
    # Summary by drug
    print(f"\n{'=' * 80}")
    print("SUMMARY BY DRUG")
    print(f"{'=' * 80}\n")
    
    drug_summary = df_diff.groupby('drug').size().sort_values(ascending=False)
    for drug, count in drug_summary.items():
        print(f"{drug}: {count} differences")
    
    # Summary by ethnicity
    print(f"\n{'=' * 80}")
    print("SUMMARY BY ETHNICITY")
    print(f"{'=' * 80}\n")
    
    ethnicity_summary = df_diff.groupby('ethnicity').size().sort_values(ascending=False)
    for ethnicity, count in ethnicity_summary.items():
        print(f"{ethnicity}: {count} differences")
    
    # Save to CSV for easier inspection
    output_path = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\drug_quantity_differences.csv"
    # df_diff.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n{'=' * 80}")
    print(f"Results saved to: {output_path}")
    print(f"{'=' * 80}")
    
else:
    print("\n✅ No differences found in drug quantities!")

## check errors

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error

# =========================
# Configuration
# =========================
COLUMN_TO_COMPARE = "age"  # Change to "age", "role", "punishment_months", etc.
FILTER_ETHNICITY = "Jewish"  # Set to None to compare all ethnicities
IS_NUMERIC = False
# Criminal_record,role, laboratory

# =========================
# 1) Load
# # =========================
def compute_punishment_months(df):
    df = df.copy()
    custodial_mask = (
        df['sentence_type'].astype(str).str.contains('עבודות שירות|בפועל', regex=True, na=False)
    )
    df['punishment_months'] = np.where(
        custodial_mask,
        df['sentence_months'].fillna(0),
        0
    )
    return df
    
df1 = pd.read_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\GT.csv",
    encoding="utf-8-sig"
)
df2 = pd.read_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\PRED.csv",
    encoding="utf-8-sig"
)
print(f"Loaded GT: {len(df1)} rows")
print(f"Loaded PRED: {len(df2)} rows")

# df1 = compute_punishment_months(df1)
# df2 = compute_punishment_months(df2)

# =========================
# 2) Align on original_case
# =========================
def align_on_case(df1_g, df2_g, key="original_case"):
    common = np.intersect1d(
        df1_g[key].dropna().unique(),
        df2_g[key].dropna().unique()
    )
    
    df1_a = (df1_g[df1_g[key].isin(common)]
             .sort_values(key)
             .reset_index(drop=True))
    
    df2_a = (df2_g[df2_g[key].isin(common)]
             .sort_values(key)
             .reset_index(drop=True))
    
    return df1_a, df2_a

df1, df2 = align_on_case(df1, df2, key="original_case")
print(f"\nAligned: {len(df1)} rows")

# =========================
# 3) Compare column
# =========================
if IS_NUMERIC:
    gt_values = pd.to_numeric(df1[COLUMN_TO_COMPARE], errors="coerce")
    pred_values = pd.to_numeric(df2[COLUMN_TO_COMPARE], errors="coerce")
    diff_mask = (gt_values.isna() ^ pred_values.isna()) | (gt_values.ne(pred_values) & gt_values.notna() & pred_values.notna())
else:
    # Categorical comparison
    gt_values = df1[COLUMN_TO_COMPARE].fillna("").astype(str).str.strip()
    pred_values = df2[COLUMN_TO_COMPARE].fillna("").astype(str).str.strip()
    diff_mask = gt_values.str.lower() != pred_values.str.lower()

# Build differences table
diff_df = pd.DataFrame({
    "original_case": df1.loc[diff_mask, "original_case"].values,
    "Defendants_Ethnic": df1.loc[diff_mask, "Defendants_Ethnic"].values,
    f"GT_{COLUMN_TO_COMPARE}": gt_values.loc[diff_mask].values,
    f"PRED_{COLUMN_TO_COMPARE}": pred_values.loc[diff_mask].values,
})

# =========================
# 4) Overall Statistics
# =========================

print("\n" + "="*70)
print(f"OVERALL STATISTICS FOR '{COLUMN_TO_COMPARE}'")
print("="*70)

# Calculate F1 for all data
y_true_all = gt_values.fillna("MISSING")
y_pred_all = pred_values.fillna("MISSING")
f1_all = f1_score(y_true_all, y_pred_all, average='macro', zero_division=0)

print(f"\nF1 Score (All data): {f1_all:.4f}")
print(f"Total cases: {len(df1)}")
print(f"Total differences: {len(diff_df)} ({100*len(diff_df)/len(df1):.1f}%)")
print(f"Total matches: {len(df1) - len(diff_df)} ({100*(len(df1)-len(diff_df))/len(df1):.1f}%)")

# calculate MAE and RMSE if numeric
# make it to numeric if possible

    
# Unique values
print(f"\nUnique GT values: {sorted(gt_values.unique())}")
print(f"Unique PRED values: {sorted(pred_values.unique())}")

# Confusion Matrix
print("\nConfusion Matrix (All data):")
cm = confusion_matrix(y_true_all, y_pred_all)
unique_labels = sorted(set(y_true_all) | set(y_pred_all))
cm_df = pd.DataFrame(cm, index=unique_labels, columns=unique_labels)
print(cm_df)

# Classification Report
print("\nClassification Report (All data):")
print(classification_report(y_true_all, y_pred_all, zero_division=0))

# =========================
# 5) By Ethnicity
# =========================
for ethnicity in df1['Defendants_Ethnic'].unique():
    if pd.isna(ethnicity):
        continue
    
    print("\n" + "="*70)
    print(f"ETHNICITY: {ethnicity}")
    print("="*70)
    
    mask = df1['Defendants_Ethnic'] == ethnicity
    y_true_eth = gt_values[mask].fillna("MISSING")
    y_pred_eth = pred_values[mask].fillna("MISSING")
    
    if len(y_true_eth) == 0:
        continue
    
    f1_eth = f1_score(y_true_eth, y_pred_eth, average='macro', zero_division=0)
    
    diff_eth = diff_df[diff_df['Defendants_Ethnic'] == ethnicity]
    n_total = len(y_true_eth)
    n_diff = len(diff_eth)
    n_match = n_total - n_diff
    
    print(f"F1 Score: {f1_eth:.4f}")
    print(f"Total cases: {n_total}")
    print(f"Differences: {n_diff} ({100*n_diff/n_total:.1f}%)")
    print(f"Matches: {n_match} ({100*n_match/n_total:.1f}%)")
    
    # Value distribution
    print(f"\nGT Value Counts:")
    print(y_true_eth.value_counts().sort_index())
    print(f"\nPRED Value Counts:")
    print(y_pred_eth.value_counts().sort_index())

# =========================
# 6) Detailed Differences (Filtered by Ethnicity if specified)
# =========================
if FILTER_ETHNICITY:
    diff_df_filtered = diff_df[diff_df["Defendants_Ethnic"] == FILTER_ETHNICITY].copy()
    ethnicity_label = f"{FILTER_ETHNICITY} only"
else:
    diff_df_filtered = diff_df.copy()
    ethnicity_label = "all ethnicities"

print("\n" + "="*70)
print(f"DETAILED DIFFERENCES ({ethnicity_label})")
print("="*70)
print(f"\nTotal differences: {len(diff_df_filtered)}")

if len(diff_df_filtered) > 0:
    print("\nShowing all differences:\n")
    for idx, row in diff_df_filtered.sort_values("original_case").iterrows():
        print(f"Ethnicity: {row['Defendants_Ethnic']}")
        print(f"Case ID: {row['original_case']}")
        print(f"  GT:   '{row[f'GT_{COLUMN_TO_COMPARE}']}'")
        print(f"  PRED: '{row[f'PRED_{COLUMN_TO_COMPARE}']}'")
        print()
else:
    print("No differences found.")

FILTER_ETHNICITY = "Not Jewish"  # Set to None to compare all ethnicities
if FILTER_ETHNICITY:
    diff_df_filtered = diff_df[diff_df["Defendants_Ethnic"] == FILTER_ETHNICITY].copy()
    ethnicity_label = f"{FILTER_ETHNICITY} only"
else:
    diff_df_filtered = diff_df.copy()
    ethnicity_label = "all ethnicities"

print("\n" + "="*70)
print(f"DETAILED DIFFERENCES ({ethnicity_label})")
print("="*70)
print(f"\nTotal differences: {len(diff_df_filtered)}")

if len(diff_df_filtered) > 0:
    print("\nShowing all differences:\n")
    for idx, row in diff_df_filtered.sort_values("original_case").iterrows():
        print(f"Ethnicity: {row['Defendants_Ethnic']}")
        print(f"Case ID: {row['original_case']}")
        print(f"  GT:   '{row[f'GT_{COLUMN_TO_COMPARE}']}'")
        print(f"  PRED: '{row[f'PRED_{COLUMN_TO_COMPARE}']}'")
        print()
else:
    print("No differences found.")
print("="*70)
print("Analysis complete!")
print("="*70)


## check punishment ranges

In [ ]:
import os
import json
import re
import pandas as pd
from openai import OpenAI

# =========================
# Configuration
# =========================
CSV_DIRS = [
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\CSV_OUTPUT",
]

GT_PATH = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\GT.csv"
PRED_PATH = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\PRED.csv"
OUTPUT_PATH = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\merged_data.csv"

PUNISHMENT_PATTERN = r"(מתח. ה?עו?ני?ש|מתחם)"
REQUIRED_PARTS = [
    "מתחמי ענישה", "אחידות בענישה", "מתחם הענישה", "מתחם ענישה",
    "מתחמי הענישה", "מתחם העונש", "מתחם עונש"
]

# =========================
# Helper Functions
# =========================
def split_into_sentences(text: str) -> list[str]:
    if not isinstance(text, str):
        return []
    s = re.split(r'[.!?]+\s+', text)
    return [x.strip() for x in s if x.strip()]

def process_text_for_gpt(text: str) -> list[str]:
    if not text or not text.strip():
        return []
    sentences = split_into_sentences(text)
    out = []
    seen = set()

    for i, sent in enumerate(sentences):
        matches_regex = bool(re.search(PUNISHMENT_PATTERN, sent))
        matches_required = any(p in sent for p in REQUIRED_PARTS)
        matches_range = bool(re.search(r'\d+\s*[-–]\s*\d+\s*(חודש|חודשים|חודשי|שנה|שנים|שנת)', sent))

        if matches_regex or matches_required or matches_range:
            if i > 0:
                combined = sentences[i-1] + " " + sent
            elif i < len(sentences)-1:
                combined = sent + " " + sentences[i+1]
            else:
                combined = sent

            norm = " ".join(combined.split())
            if norm not in seen:
                seen.add(norm)
                out.append(combined)

    return out

def build_verdict_to_texts(csv_dir: str) -> dict[str, list[str]]:
    """Build dictionary mapping verdict ID to list of relevant text snippets"""
    verdict_to_texts = {}
    files = [f for f in os.listdir(csv_dir) if f.endswith(".csv")]

    for f in files:
        verdict = f[:-4]  # Remove .csv extension to get verdict ID
        path = os.path.join(csv_dir, f)
        try:
            df = pd.read_csv(path, encoding="utf-8-sig")
        except Exception as e:
            print(f"Error reading {f}: {e}")
            continue

        collected = []
        for _, row in df.iterrows():
            # Pick one text field if exists
            t = None
            for c in ["text", "sentence", "content", "paragraph"]:
                if c in df.columns and pd.notna(row.get(c)):
                    s = str(row.get(c)).strip()
                    if s:
                        t = s
                        break
            if not t:
                continue

            # If "part" is relevant, keep raw, else sentence-filter
            part_val = str(row.get("part")).strip() if "part" in df.columns and pd.notna(row.get("part")) else ""
            part_matches_required = any(p in part_val for p in REQUIRED_PARTS)

            if part_matches_required:
                processed = [t]
            else:
                processed = process_text_for_gpt(t)

            collected.extend(processed)

        # Dedup
        seen = set()
        uniq = []
        for x in collected:
            norm = " ".join(str(x).split())
            if norm and norm not in seen:
                seen.add(norm)
                uniq.append(x)

        verdict_to_texts[verdict] = uniq

    return verdict_to_texts

# =========================
# Main Processing
# =========================
def main():
    print("=" * 60)
    print("Building verdict_to_texts from CSV directories...")
    print("=" * 60)
    
    # Build verdict_to_texts from all CSV directories
    all_verdict_texts = {}
    for csv_dir in CSV_DIRS:
        if not os.path.exists(csv_dir):
            print(f"Warning: Directory not found: {csv_dir}")
            continue
        print(f"\nProcessing directory: {csv_dir}")
        verdict_texts = build_verdict_to_texts(csv_dir)
        all_verdict_texts.update(verdict_texts)
        print(f"  Found {len(verdict_texts)} verdicts with {sum(len(v) for v in verdict_texts.values())} text snippets")
    
    print(f"\nTotal verdicts with text: {len(all_verdict_texts)}")
    print(f"Total text snippets: {sum(len(v) for v in all_verdict_texts.values())}")
    
    # Load GT and PRED data
    print("\n" + "=" * 60)
    print("Loading GT and PRED data...")
    print("=" * 60)
    
    try:
        df_gt = pd.read_csv(GT_PATH, encoding="utf-8-sig")
        print(f"GT loaded: {len(df_gt)} rows")
    except Exception as e:
        print(f"Error loading GT: {e}")
        return
    
    try:
        df_pred = pd.read_csv(PRED_PATH, encoding="utf-8-sig")
        print(f"PRED loaded: {len(df_pred)} rows")
    except Exception as e:
        print(f"Error loading PRED: {e}")
        return
    
    # Ensure both have 'original_case' column
    if 'original_case' not in df_gt.columns or 'original_case' not in df_pred.columns:
        print("Error: Both GT and PRED must have 'original_case' column")
        return
    
    # Merge GT and PRED on original_case
    df_merged = pd.merge(
        df_gt, 
        df_pred, 
        on='original_case', 
        how='inner',
        suffixes=('_gt', '_pred')
    )
    print(f"\nMerged GT and PRED: {len(df_merged)} rows")
    
    # Create final dataset
    print("\n" + "=" * 60)
    print("Creating final dataset...")
    print("=" * 60)
    
    rows = []
    matched_verdicts = 0
    unmatched_verdicts = 0
    
    for _, row in df_merged.iterrows():
        case_id = str(row['original_case']).strip()
        
        # Get text snippets for this verdict
        text_snippets = all_verdict_texts.get(case_id, [])
        
        if text_snippets:
            matched_verdicts += 1
            # Join all text snippets with double newline
            combined_text = "\n\n".join(text_snippets)
        else:
            unmatched_verdicts += 1
            combined_text = ""
        
        # Extract GT and PRED values for both min and max punishment
        gt_min = row.get('min_punishment_gt', row.get('min_punishment', ''))
        gt_max = row.get('max_punishment_gt', row.get('max_punishment', ''))
        pred_min = row.get('min_punishment_pred', row.get('min_punishment', ''))
        pred_max = row.get('max_punishment_pred', row.get('max_punishment', ''))
        
        rows.append({
            'original_case': case_id,
            'text': combined_text,
            'gt_min': gt_min,
            'gt_max': gt_max,
            'pred_min': pred_min,
            'pred_max': pred_max
        })
    
    # Create DataFrame
    df_final = pd.DataFrame(rows)
    
    print(f"\nFinal dataset statistics:")
    print(f"  Total rows: {len(df_final)}")
    print(f"  Rows with text: {(df_final['text'] != '').sum()}")
    print(f"  Rows without text: {(df_final['text'] == '').sum()}")
    print(f"  Matched verdicts: {matched_verdicts}")
    print(f"  Unmatched verdicts: {unmatched_verdicts}")
    
    # Save to CSV
    print("\n" + "=" * 60)
    print(f"Saving to: {OUTPUT_PATH}")
    print("=" * 60)
    
    # df_final.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
    print("Done!")
    
    # Show sample
    print("\n" + "=" * 60)
    print("Sample rows (first 3):")
    print("=" * 60)
    for i, row in df_final.head(3).iterrows():
        print(f"\nRow {i+1}:")
        print(f"  Case ID: {row['original_case']}")
        print(f"  Text length: {len(row['text'])} chars")
        print(f"  Text preview: {row['text'][:100]}..." if row['text'] else "  Text: (empty)")
        print(f"  GT Min: {row['gt_min']}")
        print(f"  GT Max: {row['gt_max']}")
        print(f"  PRED Min: {row['pred_min']}")
        print(f"  PRED Max: {row['pred_max']}")

if __name__ == "__main__":
    main()

## llm play

In [ ]:
FEW_SHOT_EXAMPLES = """
מטרה: לזהות את המשפט שבו השופט/ת מכריז/ת על מתחם הענישה בתיק הנוכחי, ולחלץ את גבולות המתחם בחודשי מאסר בפועל.

דוגמה 1:
טקסט: "לאחר ששקלתי את כל הנסיבות, אני קובע כי מתחם העונש ההולם נע בין 12 חודשי מאסר ל-24 חודשי מאסר."
מתחם: 12 - 24

דוגמה 2:
טקסט: "הנאשם הורשע בעבירה של החזקת נשק. הוא ירצה 6 חודשי מאסר."
מתחם: אין

דוגמה 3:
טקסט: "לטענת המדינה מתחם העונש ההולם נמצא בין שנתיים עד ארבע שנות מאסר. מכל האמור הגעתי למסקנה שמתחם העונש ההולם נמצא בין 9 חודשי מאסר בפועל עד 3 שנות מאסר בפועל."
מתחם: 9 - 36

דוגמה 4:
טקסט: "ע''פ 2482/22 מדינת ישראל נ' קדורה: נקבע מתחם הנע בין 10-36 חודשי מאסר. לאחר בחינת כלל השיקולים מצאתי כי יש לקבוע בנסיבות מקרה זה מתחם ענישה של 24-40 חודשי מאסר בפועל."
מתחם: 24 - 40

דוגמה 5 (מתחמים מרובים מאותו תיק):
טקסט: "אני קובעת כי מתחם הענישה באישום הראשון נע בין שבע לבין עשר שנות מאסר בפועל; לאישום השני מאסר בפועל בין 36 לבין 60 חודשים."
מתחם: 84 - 120

דוגמה 6 (מתחמים מתיקים שונים - לקחת רק את המתחם העיקרי):
טקסט: "לאור כל האמור אני סבורה שמתחמי הענישה שיש לקבוע במקרה זה הם כדלקמן: בתיק הנשק – מתחם כולל לשני האישומים הנע בין שתיים לארבע שנות מאסר. בתיק הסמים – מתחם כולל שבין שנה לשנתיים מאסר."
מתחם: 24 - 48

דוגמה 7 (סטטוטורי מול מתחם שנקבע בפועל):
טקסט: "מתחם העונש הינו בין 30 ל-60 חודשי מאסר... בהתאם לתיקון 113... סבורני כי מתחם העונש ההולם הינו החל מ-18 ועד 42 חודשי מאסר."
מתחם: 18 - 42

דוגמה 8 (גבול תחתון לא מספרי → ריק!):
טקסט: "לאור כל זאת, אני בדעה כי מתחם הענישה... נע בין עונש מאסר קצר שניתן לשאת בעבודות שירות ועד 12 חודשי מאסר בפועל."
מתחם: ריק - 12

דוגמה 9 (אימוץ מפורש של הצעת צד):
טקסט: "התביעה ביקשה לקבוע מתחם 8-18 חודשי מאסר. מתחם זה נמצא הולם והוא יפה לענייננו."
מתחם: 8 - 18

דוגמה 10 (חודשים + שנים בסוגריים → החודשים קובעים):
טקסט: "אני קובעת שמתחם העונש הראוי נע בין 54-78 חודשי מאסר (4.5 עד 6.5 שנים)."
מתחם: 54 - 78

דוגמה 11 (גבול תחתון לא-משמורת → 0):
טקסט: "מתחם העונש ההולם נע ממאסר על תנאי ועד 12 חודשי מאסר בפועל."
מתחם: 0 - 12
"""


def build_prompt(text: str) -> str:
    return f"""{FEW_SHOT_EXAMPLES}

כעת נתח את הטקסט הבא בזהירות רבה:

טקסט: "{text}"

מטרה:
לחלץ את מתחם הענישה הסופי (מתחם הולם) שקבע בית המשפט בתיק זה בלבד.

הוראות חשיבה (בצע את הניתוח הזה לפני מתן התשובה):
1. **סריקה:** זהה את כל הטווחים המספריים בטקסט (חודשים ושנים).
2. **סינון ראשוני (שלילה):**
   - האם המספרים מופיעים במשפט המתאר תיק אחר? (חפש מילות מפתח: "רע\"פ", "ע\"פ", "ת\"פ", "בפרשת...", "באותו עניין", "בית המשפט המחוזי קבע" בהקשר של תקדים). -> אם כן, התעלם!
   - האם המספרים הם חלק מטיעוני הצדדים? ("עתרה", "ביקשה", "לטענת", "סבורה"). -> סמן כ"מועמד להדחה" (אלא אם יאושר בשלב הבא).

3. **זיהוי הקביעה הקובעת (היררכיה):**
   - **עדיפות א' - קביעה ישירה:** חפש ביטויים בגוף ראשון כמו "אני קובע", "לדידי", "סבורני", "מצאתי לנכון", "לוז המתחם", "המתחם בתיק זה".
   - **עדיפות ב' - אימוץ (JUDGE_ADOPTION):** חפש ביטויי הסכמה מפורשים ("מקובל עלי", "יפה לענייננו", "מאמץ את עמדת...", "המתחם שהוצע... הולם").
     * אם נמצא ביטוי כזה, חפש את המספרים המוצמדים אליו או אלו שהופיעו במשפט *שמיד לפניו*.
     * בדוק שהאימוץ מתייחס למתחם הענישה ולא לעונש הספציפי.

4. **עיבוד סופי:**
   - המר שנים לחודשים.
   - אם נותרו מספר מתחמים לאישומים שונים בתיק הנוכחי -> קח את הגבול העליון הגבוה ביותר ואת הגבול התחתון הגבוה ביותר ללא סכימה של עונשים.

דגשים קריטיים למניעת טעויות:
* **כלל ה"לדידי":** המילה "לדידי" (לדעתי) היא האינדיקטור החזק ביותר לקביעת השופט. היא גוברת על הכל.
* **זהירות מציטוטים:** אם משפט מסתיים ב"באותו מקרה" – זהו תקדים חיצוני. מחק אותו מיד.
* **אימוץ:** אם השופט כותב "המתחם שהציעה המאשימה מקובל עלי" – קח את המספרים מטיעוני המאשימה.

פלט סופי (3 שורות בדיוק):
משפט רלוונטי: [הציטוט המדויק שנבחר. במקרה של אימוץ, צטט את משפט האימוץ + המשפט עם המספרים]
מתחם: [X - Y] (בחודשים בלבד. לגבי הגבול התחתון X: אם כתוב רק "של"צ"/"מאסר על תנאי" -> כתוב 0. אם מוזכר "מאסר בפועל" ללא מספר (גם אם ביחד עם של"צ/תנאי) -> כתוב "ריק". אם אין מספר כלל -> "ריק")
רמת ביטחון: [גבוהה/בינונית/נמוכה]
""".strip()

In [ ]:
import pandas as pd
df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\CSV_OUTPUT\SH-15-03-13289-44.csv", encoding="utf-8-sig")
text = ""
for para in df['text']:
	text += para + "\n"
	
all_drugs_prompt = build_prompt(text)

In [ ]:
from openai import OpenAI
client = OpenAI()  # reads OPENAI_API_KEY from the environment

response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "אתה עוזר משפטי מומחה בניתוח פסקי דין בעברית, מתמחה בעבירות סמים."},
        {"role": "user", "content": all_drugs_prompt}
    ]
)
result = response.choices[0].message.content
result


## plot the results

In [ ]:
import ast
import re
import numpy as np
import pandas as pd

# =========================
# CONSTANTS
# =========================
SIGNS = ['sign_a_non_personal', 'sign_a_personal', 'sign_b']

drug_cols = [
    'lsd', 'methamphetamine', 'ayahuasca', 'cathinone', 'ketamine',
    'hashish', 'methylmethcathinone', 'cannabis_plants', 'cannabis',
    'mdma', 'cocaine', 'heroin'
]

DRUG_COLS = [
    'lsd', 'methamphetamine', 'ayahuasca', 'cathinone', 'ketamine',
    'hashish', 'methylmethcathinone', 'cannabis_plants', 'cannabis',
    'mdma', 'cocaine', 'heroin'
]

# =========================
# 1) DEDUPLICATION BY SIGN PRIORITY
# sign_b > sign_a_personal > sign_a_non_personal
# =========================
def dedupe_drug_cell(cell):
    if cell is None or (isinstance(cell, float) and np.isnan(cell)):
        return cell

    if isinstance(cell, str):
        try:
            cell = ast.literal_eval(cell)
        except Exception:
            return cell

    if not isinstance(cell, dict):
        return cell

    def canon(x):
        if x is None:
            return ""
        s = str(x)
        s = re.sub(r"\s+", "", s)
        s = s.replace("״", '"').replace("׳", "'")
        s = re.sub(r"(?<=\d),(?=\d)", ".", s)  # 0,5 -> 0.5
        return s

    non_p = cell.get('sign_a_non_personal', []) or []
    pers  = cell.get('sign_a_personal', []) or []
    b     = cell.get('sign_b', []) or []

    non_p_map = {canon(x): x for x in non_p}
    pers_map  = {canon(x): x for x in pers}
    b_map     = {canon(x): x for x in b}

    # remove duplicates from non_personal if exist in higher priority
    for k in list(non_p_map.keys()):
        if k in pers_map or k in b_map:
            non_p_map.pop(k, None)

    cell['sign_a_non_personal'] = list(non_p_map.values())
    cell['sign_a_personal'] = list(pers_map.values())
    cell['sign_b'] = list(b_map.values())

    return cell

def remove_similar_quantities(quantity_list, tolerance_percent=0.1):
    """
    Remove quantities that are very close to each other (likely duplicates with different precision)
    ומ
    Args:
        quantity_list: list of quantity strings like ['8.717-ק"ג', '8.7-ק"ג']
        tolerance_percent: if two quantities differ by less than this %, consider them duplicates
    
    Returns:
        filtered list with similar quantities removed (keeps the one with more precision)
    """
    if not quantity_list or len(quantity_list) <= 1:
        return quantity_list
    
    # Parse each quantity into (amount, unit, original_string, precision)
    parsed = []
    for q_str in quantity_list:
        if not q_str:
            continue
        
        # Clean up
        clean_str = q_str.replace('\\"', '"').replace("\\'", "'")
        clean_str = clean_str.replace('״', '"').replace('׳', "'")
        
        # Extract amount and unit
        match = re.match(r'^\[?([0-9.,]+)\s*-\s*([^\]]+)\]?$', clean_str.strip())
        if match:
            amount_str = match.group(1).replace(',', '.')
            unit = match.group(2).strip()
            try:
                amount = float(amount_str)
                # Count decimal places for precision
                precision = len(amount_str.split('.')[-1]) if '.' in amount_str else 0
                parsed.append({
                    'amount': amount,
                    'unit': unit,
                    'original': q_str,
                    'precision': precision
                })
            except ValueError:
                parsed.append({'original': q_str})  # Keep unparseable as-is
    
    if not parsed:
        return quantity_list
    
    # Group by unit
    by_unit = {}
    unparseable = []
    
    for item in parsed:
        if 'amount' in item:
            unit = item['unit']
            if unit not in by_unit:
                by_unit[unit] = []
            by_unit[unit].append(item)
        else:
            unparseable.append(item['original'])
    
    # For each unit, remove similar quantities
    result = []
    for unit, items in by_unit.items():
        if len(items) == 1:
            result.append(items[0]['original'])
            continue
        
        # Sort by amount
        items_sorted = sorted(items, key=lambda x: x['amount'])
        
        # Mark items to keep
        keep = [True] * len(items_sorted)
        
        for i in range(len(items_sorted)):
            if not keep[i]:
                continue
            
            for j in range(i + 1, len(items_sorted)):
                if not keep[j]:
                    continue
                
                amt_i = items_sorted[i]['amount']
                amt_j = items_sorted[j]['amount']
                
                # Calculate percentage difference
                avg = (amt_i + amt_j) / 2
                diff_percent = abs(amt_i - amt_j) / avg * 100
                if diff_percent <= tolerance_percent:
                    # print(f"first = {amt_i}, second = {amt_j}, list = {quantity_list}")
                    # Keep the one with more precision (more decimal places)
                    if items_sorted[i]['precision'] >= items_sorted[j]['precision']:
                        keep[j] = False
                    else:
                        keep[i] = False
                        break
        
        # Add kept items
        for i, item in enumerate(items_sorted):
            if keep[i]:
                result.append(item['original'])
    
    result.extend(unparseable)
    return result

# =========================
# 2) EXTRACT ALL QUANTITIES (AFTER DEDUPE)
# =========================
def extract_all_quantities(cell):
    if cell is None or (isinstance(cell, float) and np.isnan(cell)):
        return []

    if isinstance(cell, str):
        try:
            cell = ast.literal_eval(cell)
        except Exception:
            return []

    if not isinstance(cell, dict):
        return []

    quantities = []
    for sign in SIGNS:
        vals = cell.get(sign, [])
        if isinstance(vals, list):
            quantities.extend(vals)

    return quantities

# =========================
# 3) NORMALIZE QUANTITIES (SUM)
# =========================
def normalize_drug_quantity(quantity_str):
    if pd.isna(quantity_str) or not str(quantity_str).strip():
        return ""

    quantity_str = str(quantity_str).strip()
    total_grams = 0.0
    total_tablets = 0.0
    total_plants = 0.0
    has_valid = False

    # Clean up escaped quotes first
    quantity_str = quantity_str.replace('\\"', '"').replace("\\'", "'")
    quantity_str = quantity_str.replace('״', '"').replace('׳', "'")

    bracket_content = re.findall(r'\[(.*?)\]', quantity_str)
    all_pairs = []

    for content in bracket_content:
        parts = content.split(',')
        for part in parts:
            part = part.strip()
            match = re.match(r'^([0-9.,]+)\s*-\s*(.+)$', part)
            if match:
                all_pairs.append((match.group(1), match.group(2)))

    if not all_pairs:
        pattern = r'\[([0-9.,]+)-([^\]]+)\]'
        all_pairs = re.findall(pattern, quantity_str)
    
    # If still no pairs found, try to parse without brackets
    if not all_pairs:
        # Handle formats like: 27.5-ק"ג or [27.5-ק"ג]
        pattern = r'([0-9.,]+)\s*-\s*([^\[\],]+)'
        all_pairs = re.findall(pattern, quantity_str)

    for amount_str, unit in all_pairs:
        unit = unit.strip()
        amount_str = amount_str.replace(',', '.').strip()

        try:
            amount = float(amount_str)
            has_valid = True
        except ValueError:
            continue

        if unit in ['ק"ג', 'קילוגרם', 'קילו', 'kg', 'קג']:
            total_grams += amount * 1000
        elif unit in ['גרם', 'gram', 'grams', 'gr', 'g']:
            total_grams += amount
        elif unit in ['מ"ג', 'מיליגרם', 'mg']:
            total_grams += amount / 1000
        elif unit in ['טבליות', 'כדורים', 'יחידות', 'tablets', 'pills', 'בולים', 'כדור', 'טבליה']:
            total_tablets += amount
        elif unit in ['שתילים', 'שתיל', 'plants', 'צמחים']:
            total_plants += amount

    if not has_valid:
        return ""

    parts = []
    if total_grams > 0:
        parts.append(f"[{int(total_grams) if total_grams.is_integer() else round(total_grams, 2)}-גרם]")
    if total_tablets > 0:
        parts.append(f"[{int(total_tablets)}-טבליות]")
    if total_plants > 0:
        parts.append(f"[{int(total_plants)}-שתילים]")

    return ", ".join(parts) if parts else ""

def normalize_from_list(quantity_list):
    if quantity_list is None or not isinstance(quantity_list, list) or len(quantity_list) == 0:
        return ""
    
    # Remove similar quantities first
    filtered_list = remove_similar_quantities(quantity_list, tolerance_percent=0.01)
    
    joined = "[" + ", ".join(filtered_list) + "]"
    return normalize_drug_quantity(joined)

# =========================
# 4) EXTRACT NUMERIC VALUE
# =========================
def extract_numeric_value(normalized_str):
    if pd.isna(normalized_str) or not str(normalized_str).strip():
        return np.nan
    match = re.search(r'\[([0-9.]+)-', str(normalized_str))
    if match:
        try:
            return float(match.group(1))
        except ValueError:
            return np.nan
    return np.nan



In [ ]:
# ============================================================================
# CONSTANTS
# ============================================================================

MISSING = "__MISSING__"
SIGNS = ['sign_a_non_personal', 'sign_a_personal', 'sign_b']

numeric_cols = [
    'age', 'punishment_months', 
    'min_punishment', 'max_punishment'
]

categorical_cols = [
    'confession', 'responsabillity', 'regret', 'marital_status',
    'self_use', 'laboratory', 'role', 'sold_to_agent',
    'production_offence', 'use_offence', 'importation_offence','Criminal_record','has_independent_offenses'
]

drug_cols = [
    'lsd', 'methamphetamine', 'ayahuasca', 'cathinone', 'ketamine',
    'hashish', 'methylmethcathinone', 'cannabis_plants', 'cannabis',
    'mdma', 'cocaine', 'heroin'
]

demographic_columns = ['Defendants_Ethnic']

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def compute_punishment_months(df):
    df = df.copy()
    custodial_mask = (
        df['sentence_type'].astype(str).str.contains('עבודות שירות|בפועל', regex=True, na=False)
    )
    df['punishment_months'] = np.where(
        custodial_mask,
        df['sentence_months'].fillna(0),
        0
    )
    return df

def normalize_drug_quantity(quantity_str):
    if pd.isna(quantity_str) or not str(quantity_str).strip():
        return ""
    
    quantity_str = str(quantity_str).strip()
    total_grams = 0.0
    total_tablets = 0.0
    total_plants = 0.0
    has_valid = False

    bracket_content = re.findall(r'\[(.*?)\]', quantity_str)
    all_pairs = []
    
    for content in bracket_content:
        parts = content.split(',')
        for part in parts:
            part = part.strip()
            match = re.match(r'^([0-9.,]+)\s*-\s*(.+)$', part)
            if match:
                all_pairs.append((match.group(1), match.group(2)))
    
    if not all_pairs:
        pattern = r'\[([0-9.,]+)-([^\]]+)\]'
        all_pairs = re.findall(pattern, quantity_str)

    for amount_str, unit in all_pairs:
        unit = unit.strip()
        amount_str = amount_str.replace(',', '.').strip()
        
        try:
            amount = float(amount_str)
            has_valid = True
        except ValueError:
            continue

        if unit in ['ק"ג', 'קילוגרם', 'קילו', 'kg', 'קג']:
            total_grams += amount * 1000
        elif unit in ['גרם', 'gram', 'grams', 'gr', 'g']:
            total_grams += amount
        elif unit in ['מ"ג', 'מיליגרם', 'mg']:
            total_grams += amount / 1000
        elif unit in ['טבליות', 'כדורים', 'יחידות', 'tablets', 'pills', 'בולים', 'כדור', 'טבליה']:
            total_tablets += amount
        elif unit in ['שתילים', 'שתיל', 'plants', 'צמחים']:
            total_plants += amount

    if not has_valid:
        return ""

    parts = []
    if total_grams > 0:
        parts.append(f"[{int(total_grams) if total_grams == int(total_grams) else round(total_grams, 2)}-גרם]")
    if total_tablets > 0:
        parts.append(f"[{int(total_tablets)}-טבליות]")
    if total_plants > 0:
        parts.append(f"[{int(total_plants)}-שתילים]")

    return ", ".join(parts) if parts else ""

# ============================================================================
# VISUALIZATION HELPER FUNCTIONS
# ============================================================================

def add_sig_bracket(ax, x1, x2, y, h, text, color='#0a1628', lw=1.5, fs=12):
    """Draw a significance bracket (like 'ח') between x1 and x2."""
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], color=color, lw=lw, clip_on=False)
    ax.text((x1 + x2) / 2, y + h, text, ha='center', va='bottom',
            color=color, fontsize=fs, fontweight='bold')

def get_stars(p_value):
    """Convert p-value to significance stars."""
    if pd.isna(p_value):
        return ''
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    return ''

def translate_label(label):
    """Format labels for better readability (English only)."""
    # Specific replacements
    if label == 'Not Jewish':
        return 'Arab'
    
    # General formatting
    return label.replace('_', ' ').title()

# ============================================================================
# METRICS CALCULATION FUNCTIONS
# ============================================================================

def calculate_categorical_metrics(df1, df2, col):
    """Calculate accuracy, F1, precision, and recall for a categorical column."""
    y_true = df1[col].fillna(MISSING).astype(str)
    y_pred = df2[col].fillna(MISSING).astype(str)
    
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    
    matches = (y_true == y_pred).astype(float).values
    # add number of errors and error rate
    n_errors = int((y_true != y_pred).sum())
    error_rate = n_errors / len(y_true) if len(y_true) > 0 else np.nan
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': prec,
        'recall': rec,
        'matches': matches,
        'y_true': y_true.values,
        'y_pred': y_pred.values,
        'n_errors': n_errors,
        'error_rate': error_rate,
        'n': len(y_true)
    }

def calculate_numeric_metrics(df1, df2, col, tol=0.0):
    """
    Metrics for numeric column:
    - MAE/RMSE on all valid pairs
    - error_rate: fraction of valid cases with |gt-pred| > tol
    - error_size_mae: mean |error| among error cases only
    - error_size_median: median |error| among error cases only
    - (optional) error_size_rmse: RMSE among error cases only
    """

    gt = pd.to_numeric(df1[col], errors='coerce')
    pred = pd.to_numeric(df2[col], errors='coerce')
    mask = gt.notna() & pred.notna()

    if mask.sum() == 0:
        return None

    gt_valid = gt[mask].astype(float)
    pred_valid = pred[mask].astype(float)

    diff = pred_valid - gt_valid
    abs_err = diff.abs()
    sq_err = diff.pow(2)

    # overall metrics
    mae = abs_err.mean()
    rmse = np.sqrt(sq_err.mean())

    # error cases (allow tolerance)
    err_mask = abs_err > tol
    n_valid = int(mask.sum())
    n_err = int(err_mask.sum())
    error_rate = n_err / n_valid if n_valid > 0 else np.nan

    if n_err > 0:
        error_size_mae = abs_err[err_mask].mean()
        error_size_median = abs_err[err_mask].median()
        error_size_rmse = np.sqrt(sq_err[err_mask].mean())
        # שמור גם את errors של "טעויות בלבד" ל-CI נפרד אם תרצה
        errors_only = abs_err[err_mask].values
        squared_errors_only = sq_err[err_mask].values
    else:
        error_size_mae = 0.0
        error_size_median = 0.0
        error_size_rmse = 0.0
        errors_only = np.array([])
        squared_errors_only = np.array([])

    return {
        'mae': mae,
        'rmse': rmse,
        'errors': abs_err.values,
        'squared_errors': sq_err.values,
        'n': n_valid,

        # new
        'error_rate': error_rate,
        'n_errors': n_err,
        'error_size_mae': error_size_mae,
        'error_size_median': error_size_median,
        'error_size_rmse': error_size_rmse,
        'errors_only': errors_only,
        'squared_errors_only': squared_errors_only,
    }


def calculate_combined_drug_metrics(df1, df2, drug_cols):
    """Calculate combined MAE/RMSE (Log scale) AND MAPE (Original scale)."""
    all_gt_log = []
    all_pred_log = []
    all_pe = []  # רשימה לאחוזי שגיאה
    df1 = df1.copy()  # <-- avoid mutating the original DataFrame
    df2 = df2.copy()  # <-- avoid mutating the original DataFrame
    for drug_col in drug_cols:
        df1[drug_col] = df1[drug_col].apply(dedupe_drug_cell)
        df2[drug_col] = df2[drug_col].apply(dedupe_drug_cell)
        df1[drug_col] = df1[drug_col].apply(extract_all_quantities)
        df2[drug_col] = df2[drug_col].apply(extract_all_quantities)
        df1[drug_col] = df1[drug_col].apply(normalize_from_list)
        df2[drug_col] = df2[drug_col].apply(normalize_from_list)
        df1[drug_col] = df1[drug_col].apply(extract_numeric_value)
        df2[drug_col] = df2[drug_col].apply(extract_numeric_value)

        gt_values = pd.to_numeric(df1[drug_col], errors="coerce")
        pred_values = pd.to_numeric(df2[drug_col], errors="coerce")
        # Valid mask where both have values and both > 0
        valid_mask = (gt_values > 0) & (pred_values > 0)
        
        if valid_mask.sum() > 0:
            # 1. Log scale for MAE/RMSE
            all_gt_log.extend(np.log(gt_values[valid_mask]).tolist())
            all_pred_log.extend(np.log(pred_values[valid_mask]).tolist())
            
            # 2. Original scale for MAPE
            gts = gt_values[valid_mask].values
            preds = pred_values[valid_mask].values
            # חישוב אחוז השגיאה: |(Pred - GT) / GT| * 100
            pes = (np.abs(preds - gts) / gts) * 100
            all_pe.extend(pes.tolist())
    
    if len(all_gt_log) == 0:
        return None
    
    # חישוב מדדי הלוג (רגיל)
    all_gt_log = np.array(all_gt_log)
    all_pred_log = np.array(all_pred_log)
    
    diff = all_pred_log - all_gt_log
    abs_err = np.abs(diff)
    sq_err = diff ** 2
    
    mae = abs_err.mean()
    rmse = np.sqrt(sq_err.mean())
    
    # חישוב מדד MAPE (חדש)
    all_pe = np.array(all_pe)
    mape = all_pe.mean()
    
    # error cases
    err_mask = abs_err > 0.0
    n_valid = len(all_gt_log)
    n_err = int(err_mask.sum())
    error_rate = n_err / n_valid if n_valid > 0 else np.nan
    
    if n_err > 0:
        error_size_mae = abs_err[err_mask].mean()
        error_size_median = np.median(abs_err[err_mask])
        error_size_rmse = np.sqrt(sq_err[err_mask].mean())
        errors_only = abs_err[err_mask]
        squared_errors_only = sq_err[err_mask]
    else:
        error_size_mae = 0.0
        error_size_median = 0.0
        error_size_rmse = 0.0
        errors_only = np.array([])
        squared_errors_only = np.array([])
    
    return {
        'mae': mae,
        'rmse': rmse,
        'mape': mape,            # <-- הוספנו
        'pe_values': all_pe,     # <-- הוספנו ל-CI
        'errors': abs_err,
        'squared_errors': sq_err,
        'n': n_valid,
        'error_rate': error_rate,
        'n_errors': n_err,
        'error_size_mae': error_size_mae,
        'error_size_median': error_size_median,
        'error_size_rmse': error_size_rmse,
        'errors_only': errors_only,
        'squared_errors_only': squared_errors_only,
    }

def bootstrap_ci(data, alpha=0.05, n_bootstrap=1000):
    """Calculate bootstrap confidence interval for the mean."""
    if len(data) == 0:
        return np.nan, np.nan
    
    bootstrapped = []
    n = len(data)
    
    for _ in range(n_bootstrap):
        sample = np.random.choice(data, size=n, replace=True)
        bootstrapped.append(np.mean(sample))
    
    lower = np.percentile(bootstrapped, alpha/2 * 100)
    upper = np.percentile(bootstrapped, (1 - alpha/2) * 100)
    
    return lower, upper

def bootstrap_ci_rmse(squared_errors, alpha=0.05, n_bootstrap=1000):
    """Calculate bootstrap confidence interval for RMSE."""
    if len(squared_errors) == 0:
        return np.nan, np.nan
    
    bootstrapped = []
    n = len(squared_errors)
    
    for _ in range(n_bootstrap):
        sample = np.random.choice(squared_errors, size=n, replace=True)
        bootstrapped.append(np.sqrt(np.mean(sample)))
    
    lower = np.percentile(bootstrapped, alpha/2 * 100)
    upper = np.percentile(bootstrapped, (1 - alpha/2) * 100)
    
    return lower, upper

def bootstrap_ci_f1(y_true, y_pred, alpha=0.05, n_bootstrap=1000):
    """Calculate bootstrap confidence interval for F1 score."""
    if len(y_true) == 0:
        return np.nan, np.nan
    
    bootstrapped = []
    n = len(y_true)
    indices = np.arange(n)
    
    for _ in range(n_bootstrap):
        sample_idx = np.random.choice(indices, size=n, replace=True)
        y_true_sample = y_true[sample_idx]
        y_pred_sample = y_pred[sample_idx]
        try:
            f1 = f1_score(y_true_sample, y_pred_sample, average='macro', zero_division=0)
            bootstrapped.append(f1)
        except:
            continue
    
    if len(bootstrapped) == 0:
        return np.nan, np.nan
    
    lower = np.percentile(bootstrapped, alpha/2 * 100)
    upper = np.percentile(bootstrapped, (1 - alpha/2) * 100)
    
    return lower, upper

def bootstrap_ci_precision(y_true, y_pred, alpha=0.05, n_bootstrap=1000):
    """Calculate bootstrap confidence interval for precision score."""
    if len(y_true) == 0:
        return np.nan, np.nan
    
    bootstrapped = []
    n = len(y_true)
    indices = np.arange(n)
    
    for _ in range(n_bootstrap):
        sample_idx = np.random.choice(indices, size=n, replace=True)
        y_true_sample = y_true[sample_idx]
        y_pred_sample = y_pred[sample_idx]
        try:
            prec, _, _, _ = precision_recall_fscore_support(
                y_true_sample, y_pred_sample, average='macro', zero_division=0
            )
            bootstrapped.append(prec)
        except:
            continue
    
    if len(bootstrapped) == 0:
        return np.nan, np.nan
    
    lower = np.percentile(bootstrapped, alpha/2 * 100)
    upper = np.percentile(bootstrapped, (1 - alpha/2) * 100)
    
    return lower, upper

def bootstrap_ci_recall(y_true, y_pred, alpha=0.05, n_bootstrap=1000):
    """Calculate bootstrap confidence interval for recall score."""
    if len(y_true) == 0:
        return np.nan, np.nan
    
    bootstrapped = []
    n = len(y_true)
    indices = np.arange(n)
    
    for _ in range(n_bootstrap):
        sample_idx = np.random.choice(indices, size=n, replace=True)
        y_true_sample = y_true[sample_idx]
        y_pred_sample = y_pred[sample_idx]
        try:
            _, rec, _, _ = precision_recall_fscore_support(
                y_true_sample, y_pred_sample, average='macro', zero_division=0
            )
            bootstrapped.append(rec)
        except:
            continue
    
    if len(bootstrapped) == 0:
        return np.nan, np.nan
    
    lower = np.percentile(bootstrapped, alpha/2 * 100)
    upper = np.percentile(bootstrapped, (1 - alpha/2) * 100)
    
    return lower, upper

# ============================================================================
# MAIN ANALYSIS FUNCTION
# ============================================================================

def analyze_by_demographic(df1, df2, demo_col):
    """Analyze all metrics by demographic groups."""
    
    groups = df1[demo_col].unique()
    groups = [g for g in groups if not pd.isna(g)]
    
    results = {
        'categorical': [],
        'numeric': []
    }
    
    all_groups = ['All'] + list(groups)
    
    for group in all_groups:
        if group == 'All':
            df1_g = df1
            df2_g = df2
            group_name = 'All'
        else:
            mask = df1[demo_col] == group
            df1_g = df1[mask]
            df2_g = df2[mask]
            group_name = str(group)
            
            if len(df1_g) < 10:
                continue
        
        # CATEGORICAL METRICS
        for col in categorical_cols:
            metrics = calculate_categorical_metrics(df1_g, df2_g, col)
            if metrics:
                ci_lower_acc, ci_upper_acc = bootstrap_ci(metrics['matches'])
                ci_lower_f1, ci_upper_f1 = bootstrap_ci_f1(metrics['y_true'], metrics['y_pred'])
                ci_lower_prec, ci_upper_prec = bootstrap_ci_precision(metrics['y_true'], metrics['y_pred'])
                ci_lower_rec, ci_upper_rec = bootstrap_ci_recall(metrics['y_true'], metrics['y_pred'])

                if col =='min_punishment':
                    col ='Lower Bounds'
                elif col == 'max_punishment':
                    col = 'Upper Bounds'
                elif col == "confession":
                    col = "Guilty Plea"
                elif col == "use_offence":
                    col = "possession_offence"
                elif col == "importation_offence":
                    col = "trafficking_offence"


                results['categorical'].append({
                    'group': group_name,
                    'variable': col,
                    'accuracy': metrics['accuracy'],
                    'accuracy_ci_lower': ci_lower_acc,
                    'accuracy_ci_upper': ci_upper_acc,
                    'f1': metrics['f1'],
                    'f1_ci_lower': ci_lower_f1,
                    'f1_ci_upper': ci_upper_f1,
                    'precision': metrics['precision'],
                    'precision_ci_lower': ci_lower_prec,
                    'precision_ci_upper': ci_upper_prec,
                    'recall': metrics['recall'],
                    'recall_ci_lower': ci_lower_rec,
                    'recall_ci_upper': ci_upper_rec,
                    'n': metrics['n'],
                    'n_errors': metrics['n_errors'],
                    'error_rate': metrics['error_rate']
                })
        
        # NUMERIC METRICS (Standard: Age, Punishment)
        for col in numeric_cols:
            metrics = calculate_numeric_metrics(df1_g, df2_g, col)
            if metrics:
                ci_lower_mae, ci_upper_mae = bootstrap_ci(metrics['errors'])
                ci_lower_rmse, ci_upper_rmse = bootstrap_ci_rmse(metrics['squared_errors'])
                
                err_indicator = (metrics['errors'] > 0.0).astype(float)
                er_ci_l, er_ci_u = bootstrap_ci(err_indicator)

                if len(metrics['errors_only']) > 0:
                    es_ci_l, es_ci_u = bootstrap_ci(metrics['errors_only'])
                else:
                    es_ci_l, es_ci_u = np.nan, np.nan

                if col =='min_punishment':
                    col ='Lower Bounds'
                elif col == 'max_punishment':
                    col = 'Upper Bounds'

                results['numeric'].append({
                    'group': group_name,
                    'variable': col,
                    'mae': metrics['mae'],
                    'mae_ci_lower': ci_lower_mae,
                    'mae_ci_upper': ci_upper_mae,
                    'rmse': metrics['rmse'],
                    'rmse_ci_lower': ci_lower_rmse,
                    'rmse_ci_upper': ci_upper_rmse,
                    
                    # עבור משתנים רגילים, אין MAPE
                    'mape': np.nan,
                    'mape_ci_lower': np.nan,
                    'mape_ci_upper': np.nan,

                    'n': metrics['n'],
                    'error_rate': metrics['error_rate'],
                    'error_rate_ci_lower': er_ci_l,
                    'error_rate_ci_upper': er_ci_u,
                    'n_errors': metrics['n_errors'],
                    'error_size_mae': metrics['error_size_mae'],
                    'error_size_median': metrics['error_size_median'],
                    'error_size_rmse': metrics['error_size_rmse'],
                    'error_size_ci_lower': es_ci_l,
                    'error_size_ci_upper': es_ci_u,
                })
        
        # COMBINED DRUG METRICS (Include MAPE)
        drug_metrics = calculate_combined_drug_metrics(df1_g, df2_g, drug_cols)
        if drug_metrics:
            ci_lower_mae, ci_upper_mae = bootstrap_ci(drug_metrics['errors'])
            ci_lower_rmse, ci_upper_rmse = bootstrap_ci_rmse(drug_metrics['squared_errors'])
            
            # --- חישוב CI ל-MAPE ---
            ci_lower_mape, ci_upper_mape = bootstrap_ci(drug_metrics['pe_values'])
            # -----------------------

            err_indicator = (metrics['errors'] > 0.0).astype(float)
            er_ci_l, er_ci_u = bootstrap_ci(err_indicator)

            if len(metrics['errors_only']) > 0:
                es_ci_l, es_ci_u = bootstrap_ci(metrics['errors_only'])
            else:
                es_ci_l, es_ci_u = np.nan, np.nan

            results['numeric'].append({
                'group': group_name,
                'variable': 'drug_quantity_log',  # זה השם שיופיע בגרפים של MAE/RMSE
                'mae': drug_metrics['mae'],
                'mae_ci_lower': ci_lower_mae,
                'mae_ci_upper': ci_upper_mae,
                'rmse': drug_metrics['rmse'],
                'rmse_ci_lower': ci_lower_rmse,
                'rmse_ci_upper': ci_upper_rmse,

                # --- הוספת הנתונים ל-MAPE ---
                'mape': drug_metrics['mape'],
                'mape_ci_lower': ci_lower_mape,
                'mape_ci_upper': ci_upper_mape,
                # ---------------------------

                'n': drug_metrics['n'],
                'error_rate': drug_metrics['error_rate'],
                'error_rate_ci_lower': er_ci_l,
                'error_rate_ci_upper': er_ci_u,
                'n_errors': drug_metrics['n_errors'],
                'error_size_mae': drug_metrics['error_size_mae'],
                'error_size_median': drug_metrics['error_size_median'],
                'error_size_rmse': drug_metrics['error_size_rmse'],
                'error_size_ci_lower': es_ci_l,
                'error_size_ci_upper': es_ci_u,
            })
    
    return {
        'categorical': pd.DataFrame(results['categorical']),
        'numeric': pd.DataFrame(results['numeric'])
    }

# ============================================================================
# PLOTTING FUNCTION
# ============================================================================

def plot_metric_comparison(df_results, metric_col, ci_lower_col, ci_upper_col, 
                           demo_col, ylabel, output_file):
    """Create professional bar plot with CI and p-values."""
    
    if len(df_results) == 0:
        print(f"Skipping {output_file} - no data")
        return
    
    # Get unique variables and groups
    variables = df_results['variable'].unique()
    groups = df_results['group'].unique()
    
    if len(variables) == 0 or len(groups) == 0:
        print(f"Skipping {output_file} - no variables or groups")
        return
    
    # Set up figure
    fig, ax = plt.subplots(figsize=(16, 6))
    
    # Colors
    colors = {
        'All': '#7B9E87',
        groups[1] if len(groups) > 1 else 'default1': '#B95A48',
        groups[2] if len(groups) > 2 else 'default2': '#1A2C48',
        groups[3] if len(groups) > 3 else 'default3': '#D4A574'
    }
    
    x = np.arange(len(variables))
    n_groups = len(groups)
    width = 0.75 / n_groups
    
    # Calculate global max for y-axis
    all_values = df_results[metric_col].values
    all_ci_upper = df_results[ci_upper_col].values
    valid_ci = ~np.isnan(all_ci_upper) & ~np.isnan(all_values)
    if valid_ci.any():
        global_max = np.nanmax(all_ci_upper[valid_ci])
    else:
        global_max = np.nanmax(all_values) if len(all_values) > 0 else 1
    
    # Plot bars for each group
    for i, group in enumerate(groups):
        df_group = df_results[df_results['group'] == group]
    
        values = []
        ci_lower = []
        ci_upper = []
        
        for var in variables:
            row = df_group[df_group['variable'] == var]
            if len(row) > 0:
                values.append(row[metric_col].values[0])
                ci_lower.append(row[ci_lower_col].values[0])
                ci_upper.append(row[ci_upper_col].values[0])
            else:
                values.append(np.nan)
                ci_lower.append(np.nan)
                ci_upper.append(np.nan)
        
        values = np.array(values)
        ci_lower = np.array(ci_lower)
        ci_upper = np.array(ci_upper)
        
        # Calculate error bars - ensure non-negative
        yerr_lower = np.abs(values - ci_lower)
        yerr_upper = np.abs(ci_upper - values)
        yerr_lower = np.nan_to_num(yerr_lower, nan=0.0)
        yerr_upper = np.nan_to_num(yerr_upper, nan=0.0)
        yerr = [yerr_lower, yerr_upper]
        
        offset = (i - n_groups/2) * width + width/2
        color = colors.get(group, f'C{i}')
        
        bars = ax.bar(x + offset, values, width,
                     label=translate_label(group),
                     color=color,
                     edgecolor='white',
                     linewidth=1.5)
        
        ax.errorbar(x + offset, values, yerr=yerr,
                   fmt='none', color='#2c3e50', 
                   capsize=4, capthick=1.2, elinewidth=1.2)
    
    # Calculate p-values and add significance brackets
    
    # Identify the specific groups (excluding 'All')
    subgroups = [g for g in groups if g != 'All']

    # We only perform this test if there are exactly 2 groups to compare
    if len(subgroups) == 2:
        g1_name, g2_name = subgroups[0], subgroups[1]
        
        # Find their indices for plotting brackets
        g1_idx = np.where(groups == g1_name)[0][0]
        g2_idx = np.where(groups == g2_name)[0][0]

        for var_idx, var in enumerate(variables):
            # Fetch data for Group 1
            d1 = df_results[(df_results['group'] == g1_name) & (df_results['variable'] == var)]
            # Fetch data for Group 2
            d2 = df_results[(df_results['group'] == g2_name) & (df_results['variable'] == var)]
            
            if len(d1) == 0 or len(d2) == 0:
                continue

            # Extract Stats Group 1
            m1 = d1[metric_col].values[0]
            n1 = d1['n'].values[0]
            ci_l1 = d1[ci_lower_col].values[0]
            ci_u1 = d1[ci_upper_col].values[0]

            # Extract Stats Group 2
            m2 = d2[metric_col].values[0]
            n2 = d2['n'].values[0]
            ci_l2 = d2[ci_lower_col].values[0]
            ci_u2 = d2[ci_upper_col].values[0]
            
            # Check for validity
            if pd.isna([m1, m2, ci_l1, ci_u1, ci_l2, ci_u2]).any():
                continue

            # 1. Recover Standard Error (SE) from the Bootstrap CI
            # Assuming 95% CI: width = 2 * 1.96 * SE
            se1 = (ci_u1 - ci_l1) / (2 * 1.96)
            se2 = (ci_u2 - ci_l2) / (2 * 1.96)
            
            if se1 <= 0 or se2 <= 0:
                continue

            # 2. Calculate SE for the Difference (Independent Samples)
            # This is valid because Group 1 and Group 2 are distinct (unlike Group vs All)
            se_diff = np.sqrt(se1**2 + se2**2)
            
            # 3. Calculate t-statistic
            t_stat = (m1 - m2) / se_diff
            
            # 4. Degrees of Freedom (Welch-Satterthwaite equation)
            df_freedom = ((se1**2 + se2**2)**2) / \
                         ((se1**4 / max(n1 - 1, 1)) + (se2**4 / max(n2 - 1, 1)))
            
            # 5. Calculate P-Value (Two-tailed)
            p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df_freedom))
            
            # 6. Draw Significance Bracket
            stars = get_stars(p_value)
            
            if stars:
                # Calculate bar positions
                g1_offset = (g1_idx - n_groups/2) * width + width/2
                g2_offset = (g2_idx - n_groups/2) * width + width/2
                
                x1 = var_idx + g1_offset
                x2 = var_idx + g2_offset
                
                # Determine bracket height
                y_max = max(ci_u1, ci_u2) if not np.isnan([ci_u1, ci_u2]).any() else max(m1, m2)
                bracket_y = y_max + global_max * 0.05
                bracket_h = global_max * 0.02
                
                add_sig_bracket(ax, x1, x2, bracket_y, bracket_h, stars, 
                              color='#4a4a4a', lw=1.2, fs=10)

    # ... rest of plotting code ...
    
    # Styling
    ax.set_ylim(bottom=0, top=global_max * 1.35)
    ax.set_ylabel(ylabel, fontsize=13, fontweight='bold', color='#1a1a1a', labelpad=10)
    # ax.set_title(f'{ylabel} by {translate_label(demo_col)}', 
    #             fontsize=16, fontweight='bold', color='#1a1a1a', pad=25)
    
    ax.set_xticks(x)
    ax.set_xticklabels([translate_label(v) for v in variables], 
                       fontsize=11, color='#1a1a1a', rotation=45, ha='right')
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#bdc3c7')
    ax.spines['bottom'].set_color('#bdc3c7')
    
    ax.yaxis.grid(True, linestyle='--', alpha=0.4, color='#bdc3c7')
    ax.xaxis.grid(False)
    ax.set_axisbelow(True)
    
    fig.legend(handles=ax.get_legend_handles_labels()[0],
              labels=ax.get_legend_handles_labels()[1],
              loc='upper center',
              bbox_to_anchor=(0.5, 0.96),
              ncol=n_groups,
              frameon=False,
              fontsize=11)
    
    ax.tick_params(axis='x', colors='#1a1a1a', labelsize=10, pad=8)
    ax.tick_params(axis='y', colors='#1a1a1a', labelsize=10)
    
    # Transparent background
    fig.patch.set_facecolor('none')
    ax.set_facecolor('none')
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight', transparent=True)
    plt.close()
    print(f"Saved: {output_file}")

# ============================================================================
# MAIN EXECUTION
# ============================================================================

# Load data
df1 = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\GT.csv", encoding="utf-8-sig")
df2 = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\PRED.csv", encoding="utf-8-sig")

#indf2 if for each column if the values is 1/0 replace with כן/לא
for col in categorical_cols:
    if col == 'importation_offence' or col == 'production_offence' or col == 'use_offence':
        continue
    if set(df2[col].dropna().unique()).issubset({0, 1}):
        df2[col] = df2[col].map({1: 'כן', 0: 'לא'})

# Compute punishment months
df1 = compute_punishment_months(df1)
df2 = compute_punishment_months(df2)

#remove row than in df1 that are not in df2 based on original_case
df1 = df1[df1['original_case'].isin(df2['original_case'])].copy()
def align_on_case(df1_g, df2_g, key='original_case'):
    common = np.intersect1d(df1_g[key].dropna().unique(), df2_g[key].dropna().unique())

    df1_a = (df1_g[df1_g[key].isin(common)]
             .sort_values(key)
             .reset_index(drop=True))

    df2_a = (df2_g[df2_g[key].isin(common)]
             .sort_values(key)
             .reset_index(drop=True))

    return df1_a, df2_a
df1, df2 = align_on_case(df1, df2, key='original_case')

# Analyze each demographic column
for demo_col in demographic_columns:
    print(f"\n=== Analyzing by {demo_col} ===")
    results = analyze_by_demographic(df1, df2, demo_col)
    
    # Save results to CSV
    results['categorical'].to_csv(f'llm_results/categorical_metrics_{demo_col}.csv', index=False, encoding='utf-8-sig')
    results['numeric'].to_csv(f'llm_results/numeric_metrics_{demo_col}.csv', index=False, encoding='utf-8-sig')
    
    # Create plots
    # Categorical - Accuracy
    plot_metric_comparison(results['categorical'], 'accuracy', 
                          'accuracy_ci_lower', 'accuracy_ci_upper',
                          demo_col, 'Accuracy', f'llm_results/accuracy_{demo_col}.png')
    
    # Categorical - F1
    plot_metric_comparison(results['categorical'], 'f1',
                          'f1_ci_lower', 'f1_ci_upper',
                          demo_col, 'F1 Score', f'llm_results/f1_{demo_col}.png')
    
    # Categorical - Precision
    plot_metric_comparison(results['categorical'], 'precision',
                          'precision_ci_lower', 'precision_ci_upper',
                          demo_col, 'Precision', f'llm_results/precision_{demo_col}.png')
    
    # Categorical - Recall
    plot_metric_comparison(results['categorical'], 'recall',
                          'recall_ci_lower', 'recall_ci_upper',
                          demo_col, 'Recall', f'llm_results/recall_{demo_col}.png')
    
    # Numeric - MAE (includes drug_quantity_log)
    plot_metric_comparison(results['numeric'], 'mae',
                          'mae_ci_lower', 'mae_ci_upper',
                          demo_col, 'Mean Absolute Error', f'llm_results/mae_{demo_col}.png')
    
    # Numeric - RMSE (includes drug_quantity_log)
    plot_metric_comparison(results['numeric'], 'rmse',
                          'rmse_ci_lower', 'rmse_ci_upper',
                          demo_col, 'Root Mean Squared Error', f'llm_results/rmse_{demo_col}.png')

    # Error Rate
    plot_metric_comparison(results['numeric'], 'error_rate',
                        'error_rate_ci_lower', 'error_rate_ci_upper',
                        demo_col, 'Error Rate', f'llm_results/error_rate_{demo_col}.png')

    # Error Size (MAE on error cases only)
    plot_metric_comparison(results['numeric'], 'error_size_mae',
                        'error_size_ci_lower', 'error_size_ci_upper',
                        demo_col, 'Error Size (MAE | error)', f'llm_results/error_size_{demo_col}.png')
    
    df_mape = results['numeric'].dropna(subset=['mape'])
    plot_metric_comparison(df_mape, 'mape',
                            'mape_ci_lower', 'mape_ci_upper',
                            demo_col, 'Mean Absolute Percentage Error (%)', f'llm_results/mape_{demo_col}.png')

print("\n=== Analysis complete. All results saved. ===")

## csv for whole results

In [ ]:
import pandas as pd

numeric_cols = [
    'age', 'sentence_months',
    'min_punishment', 'max_punishment', 'range_punishment'
]

categorical_cols = [
    'confession', 'responsabillity', 'regret', 'marital_status',
    'self_use', 'laboratory', 'role', 'sold_to_agent',
    'production_offence', 'use_offence', 'importation_offence', 'Criminal_record','sentence_type'
]

LABEL_COLS = numeric_cols + categorical_cols

gt_path   = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\GT.csv"
pred_path = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\PRED.csv"

df_gt = pd.read_csv(gt_path, encoding="utf-8-sig")
df_pr = pd.read_csv(pred_path, encoding="utf-8-sig")

# ---- choose the text column name ----
# change this if your text column name is different (e.g., "verdict_text", "main_text", etc.)
TEXT_COL = "original_case"

# sanity checks
missing_gt = [c for c in [TEXT_COL] + LABEL_COLS if c not in df_gt.columns]
missing_pr = [c for c in [TEXT_COL] + LABEL_COLS if c not in df_pr.columns]
if missing_gt:
    raise ValueError(f"GT missing columns: {missing_gt}")
if missing_pr:
    raise ValueError(f"PRED missing columns: {missing_pr}")

# if you assume rows align, ensure same length (and optionally same text)
if len(df_gt) != len(df_pr):
    raise ValueError(f"Row count mismatch: GT={len(df_gt)} PRED={len(df_pr)}")

# OPTIONAL (strong check): verify texts are identical row-by-row
# if not identical, comment this out and use an ID-based merge instead
if not (df_gt[TEXT_COL].fillna("") == df_pr[TEXT_COL].fillna("")).all():
    raise ValueError("Texts are not aligned row-by-row. Use an ID column merge instead.")

# ---- build long format ----
gt_long = df_gt[[TEXT_COL] + LABEL_COLS].melt(
    id_vars=[TEXT_COL],
    value_vars=LABEL_COLS,
    var_name="label",
    value_name="gt"
)

pr_long = df_pr[[TEXT_COL] + LABEL_COLS].melt(
    id_vars=[TEXT_COL],
    value_vars=LABEL_COLS,
    var_name="label",
    value_name="pred"
)

out = gt_long.merge(pr_long, on=[TEXT_COL, "label"], how="inner")

# optional: add match flag
out["match"] = out["gt"].astype(str).fillna("") == out["pred"].astype(str).fillna("")

# save
out_path = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\GT_vs_PRED_long.csv"
out.to_csv(out_path, index=False, encoding="utf-8-sig")

print("Saved:", out_path)
print("Rows:", len(out))


## check drugs amount (not total)

In [ ]:
SIGNS = ["sign_a_non_personal", "sign_a_personal", "sign_b"]


# ---------- Helpers ----------
def _extract_numeric_value(qty_str: str) -> Optional[float]:
    """
    Extract numeric value from quantity string like "81.02-גרם" or "0.30-יחידה"
    Returns None if no valid number found.
    """
    import re
    # Try to find number pattern (supports decimal)
    match = re.search(r'(\d+\.?\d*)', qty_str)
    if match:
        try:
            return float(match.group(1))
        except ValueError:
            return None
    return None


def _f1(precision: float, recall: float) -> float:
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def _safe_div(a: float, b: float) -> float:
    return a / b if b else 0.0


def _as_list(val: Any) -> List[str]:
    """Normalize input (None, string, or list) into a list of strings."""
    if val is None:
        return []
    if isinstance(val, (list, tuple)):
        return [str(v) for v in val if v is not None]
    return [str(val)] if str(val).strip() else []


# ---------- Error Analysis Classes ----------
@dataclass
class ErrorExample:
    """Single error instance with context"""
    case_id: str
    drug: str
    quantity: str
    gt_sign: Optional[str]
    pred_sign: Optional[str]
    error_type: str  # 'FP', 'FN', 'WRONG_SIGN', 'DUPLICATE'


@dataclass
class ErrorAnalysis:
    """Detailed breakdown of errors"""
    false_positives: List[ErrorExample] = field(default_factory=list)
    false_negatives: List[ErrorExample] = field(default_factory=list)
    wrong_signs: List[ErrorExample] = field(default_factory=list)
    duplicates: List[ErrorExample] = field(default_factory=list)
    
    def get_summary(self) -> Dict[str, int]:
        return {
            'total_errors': len(self.false_positives) + len(self.false_negatives) + len(self.wrong_signs) + len(self.duplicates),
            'false_positives': len(self.false_positives),
            'false_negatives': len(self.false_negatives),
            'wrong_signs': len(self.wrong_signs),
            'duplicates': len(self.duplicates)
        }
    
    def get_top_errors(self, n: int = 10) -> Dict[str, List[ErrorExample]]:
        """Get top N errors of each type"""
        return {
            'false_positives': self.false_positives[:n],
            'false_negatives': self.false_negatives[:n],
            'wrong_signs': self.wrong_signs[:n],
            'duplicates': self.duplicates[:n]
        }


# ---------- Metrics outputs ----------
@dataclass
class StrictMetrics:
    tp: int
    fp: int
    fn: int
    precision: float
    recall: float
    f1: float


@dataclass
class QuantityOnlyMetrics:
    """
    Match on (case, drug, quantity_sum) ignoring sign.
    """
    tp: int
    fp: int
    fn: int
    precision: float
    recall: float
    f1: float


@dataclass
class SignMetrics:
    """
    Evaluate sign classification for quantities that match on (case, drug).
    Includes confusion matrix.
    """
    correct: int
    total_matched_items: int  # items that matched on (case, drug, sum)
    accuracy: float
    confusion: Dict[str, Dict[str, int]]  # gt_sign -> pred_sign -> count
    
    def get_confusion_percentages(self) -> Dict[str, Dict[str, float]]:
        """Convert confusion matrix to percentages per GT class"""
        result = {}
        for gt_sign, pred_dict in self.confusion.items():
            total = sum(pred_dict.values())
            if total > 0:
                result[gt_sign] = {
                    pred_sign: round(100 * count / total, 2)
                    for pred_sign, count in pred_dict.items()
                }
            else:
                result[gt_sign] = {ps: 0.0 for ps in pred_dict.keys()}
        return result


@dataclass
class DuplicateMetrics:
    """
    Duplicate rate inside each (case, drug, sign): repeated identical qty strings.
    """
    total_groups: int
    groups_with_dups: int
    duplicate_items: int  # number of extra duplicates beyond first occurrence
    duplicate_rate_groups: float  # groups_with_dups / total_groups


@dataclass
class SpecificityMetrics:
    """
    Evaluate how well the model avoids false positives on empty cells.
    Measures: "How good is the model at NOT inventing data?"
    """
    total_empty_cells: int  # Total (case, drug, sign) cells that are empty in GT
    true_negatives: int  # Empty cells correctly left empty in PRED
    false_positives_empty: int  # Empty cells incorrectly filled in PRED
    specificity: float  # TN / (TN + FP_empty)
    
    def get_summary(self) -> Dict[str, Any]:
        return {
            'total_empty_cells': self.total_empty_cells,
            'true_negatives': self.true_negatives,
            'false_positives_empty': self.false_positives_empty,
            'specificity': round(self.specificity, 3),
            'specificity_percent': f"{self.specificity * 100:.1f}%"
        }


@dataclass
class QuantityDistanceMetrics:
    """
    Analyze numerical distance between GT and PRED quantities.
    Uses summation approach.
    """
    total_comparisons: int  # Number of (case, drug) pairs to compare
    exact_matches: int  # Quantities that match exactly
    near_misses: List[Tuple[str, str, float, float, float]]  # (case, drug, gt_val, pred_val, diff)
    mean_absolute_error: float  # Average |GT - PRED|
    median_absolute_error: float
    per_sign_metrics: Dict[str, Dict[str, Any]] = field(default_factory=dict)  # per sign category breakdown
    
    def get_summary(self) -> Dict[str, Any]:
        return {
            'total_comparisons': self.total_comparisons,
            'exact_matches': self.exact_matches,
            'exact_match_rate': _safe_div(self.exact_matches, self.total_comparisons),
            'mean_absolute_error': round(self.mean_absolute_error, 3),
            'median_absolute_error': round(self.median_absolute_error, 3),
            'near_misses_count': len(self.near_misses)
        }
    
    def get_per_sign_summary(self) -> Dict[str, Dict[str, Any]]:
        """Get distance metrics broken down by sign category"""
        return self.per_sign_metrics

    def _as_list(val: Any) -> List[str]:
        """Normalize input (None, string, or list) into a list of strings."""
        if val is None:
            return []
        if isinstance(val, (list, tuple)):
            return [str(v) for v in val if v is not None]
        return [str(val)] if str(val).strip() else []


@dataclass
class CoverageMetrics:
    """Track case and drug level coverage"""
    gt_cases: Set[str]
    pred_cases: Set[str]
    missing_cases: Set[str]  # in GT but not PRED
    extra_cases: Set[str]  # in PRED but not GT
    gt_drugs: Set[Tuple[str, str]]  # (case, drug)
    pred_drugs: Set[Tuple[str, str]]
    missing_drugs: Set[Tuple[str, str]]
    extra_drugs: Set[Tuple[str, str]]
    
    def get_summary(self) -> Dict[str, Any]:
        return {
            'case_coverage': {
                'gt_total': len(self.gt_cases),
                'pred_total': len(self.pred_cases),
                'missing': len(self.missing_cases),
                'extra': len(self.extra_cases),
                'coverage_rate': _safe_div(len(self.gt_cases & self.pred_cases), len(self.gt_cases))
            },
            'drug_coverage': {
                'gt_total': len(self.gt_drugs),
                'pred_total': len(self.pred_drugs),
                'missing': len(self.missing_drugs),
                'extra': len(self.extra_drugs),
                'coverage_rate': _safe_div(len(self.gt_drugs & self.pred_drugs), len(self.gt_drugs))
            }
        }


@dataclass
class EvaluationReport:
    strict: StrictMetrics
    quantity_only: QuantityOnlyMetrics
    sign: SignMetrics
    duplicates_pred: DuplicateMetrics
    duplicates_gt: DuplicateMetrics
    coverage: CoverageMetrics
    error_analysis: ErrorAnalysis
    quantity_distance: QuantityDistanceMetrics
    specificity: SpecificityMetrics
    
    def print_summary(self):
        """Print a comprehensive summary of the evaluation"""
        print("=" * 80)
        print("EVALUATION SUMMARY (SUMMATION METHOD)")
        print("=" * 80)
        
        print("\n--- STRICT MATCHING (case, drug, sign, quantity_sum) ---")
        print(f"Precision: {self.strict.precision:.3f} ({self.strict.tp}/{self.strict.tp + self.strict.fp})")
        print(f"Recall:    {self.strict.recall:.3f} ({self.strict.tp}/{self.strict.tp + self.strict.fn})")
        print(f"F1 Score:  {self.strict.f1:.3f}")
        print(f"TP: {self.strict.tp}, FP: {self.strict.fp}, FN: {self.strict.fn}")
        
        print("\n--- QUANTITY-ONLY MATCHING (ignoring sign, sum per case+drug) ---")
        print(f"Precision: {self.quantity_only.precision:.3f}")
        print(f"Recall:    {self.quantity_only.recall:.3f}")
        print(f"F1 Score:  {self.quantity_only.f1:.3f}")
        print(f"TP: {self.quantity_only.tp}, FP: {self.quantity_only.fp}, FN: {self.quantity_only.fn}")
        
        print("\n--- SIGN CLASSIFICATION (for matched case+drug sums) ---")
        print(f"Accuracy: {self.sign.accuracy:.3f} ({self.sign.correct}/{self.sign.total_matched_items})")
        print(f"\nConfusion Matrix (counts):")
        self._print_confusion_matrix(self.sign.confusion)
        
        print(f"\nConfusion Matrix (percentages per GT class):")
        self._print_confusion_matrix(self.sign.get_confusion_percentages(), is_percent=True)
        
        print("\n--- SPECIFICITY (Empty Cell Accuracy) ---")
        spec_summary = self.specificity.get_summary()
        print(f"Total empty cells in GT: {spec_summary['total_empty_cells']}")
        print(f"True Negatives (correctly left empty): {spec_summary['true_negatives']}")
        print(f"False Positives (incorrectly filled): {spec_summary['false_positives_empty']}")
        print(f"Specificity: {spec_summary['specificity']} ({spec_summary['specificity_percent']})")
        print(f"\nInterpretation: The model correctly avoids inventing data in {spec_summary['specificity_percent']} of empty cells.")
        
        print("\n--- DUPLICATE ANALYSIS ---")
        print(f"Prediction duplicates: {self.duplicates_pred.duplicate_rate_groups:.2%} of groups")
        print(f"  Groups with duplicates: {self.duplicates_pred.groups_with_dups}/{self.duplicates_pred.total_groups}")
        print(f"  Total duplicate items: {self.duplicates_pred.duplicate_items}")
        
        print(f"\nGround truth duplicates: {self.duplicates_gt.duplicate_rate_groups:.2%} of groups")
        print(f"  Groups with duplicates: {self.duplicates_gt.groups_with_dups}/{self.duplicates_gt.total_groups}")
        
        print("\n--- COVERAGE ANALYSIS ---")
        cov_summary = self.coverage.get_summary()
        print(f"Case coverage: {cov_summary['case_coverage']['coverage_rate']:.2%}")
        print(f"  Missing cases: {len(self.coverage.missing_cases)}")
        print(f"  Extra cases: {len(self.coverage.extra_cases)}")
        print(f"Drug coverage: {cov_summary['drug_coverage']['coverage_rate']:.2%}")
        print(f"  Missing drugs: {len(self.coverage.missing_drugs)}")
        print(f"  Extra drugs: {len(self.coverage.extra_drugs)}")
        
        print("\n--- QUANTITY DISTANCE ANALYSIS ---")
        qty_summary = self.quantity_distance.get_summary()
        print(f"Total (case, drug) comparisons: {qty_summary['total_comparisons']}")
        print(f"Exact matches: {qty_summary['exact_matches']} ({qty_summary['exact_match_rate']:.2%})")
        print(f"Mean Absolute Error: {qty_summary['mean_absolute_error']}")
        print(f"Median Absolute Error: {qty_summary['median_absolute_error']}")
        print(f"\nTop 10 Near Misses (closest incorrect predictions):")
        for i, (case, drug, gt_val, pred_val, diff) in enumerate(self.quantity_distance.near_misses[:90], 1):
            print(f"  {i}. {case} | {drug} | GT:{gt_val:.2f} PRED:{pred_val:.2f} | Diff:{diff:.3f}")
        
        print("\n--- QUANTITY DISTANCE BY SIGN CATEGORY ---")
        per_sign = self.quantity_distance.get_per_sign_summary()
        for sign_name, metrics in per_sign.items():
            print(f"\n{sign_name}:")
            print(f"  Comparisons: {metrics['comparisons']}")
            print(f"  Exact matches: {metrics['exact_matches']} ({metrics['exact_match_rate']:.2%})")
            print(f"  Mean Absolute Error: {metrics['mean_abs_error']:.3f}")
            print(f"  Median Absolute Error: {metrics['median_abs_error']:.3f}")
            if metrics['near_misses']:
                print(f"  Top 3 near misses:")
                for case, drug, gt_val, pred_val, diff in metrics['near_misses'][:3]:
                    print(f"    {case} | {drug} | GT:{gt_val:.2f} PRED:{pred_val:.2f} | Diff:{diff:.3f}")
        
        print("\n--- ERROR SUMMARY ---")
        err_summary = self.error_analysis.get_summary()
        print(f"Total errors: {err_summary['total_errors']}")
        print(f"  False positives: {err_summary['false_positives']}")
        print(f"  False negatives: {err_summary['false_negatives']}")
        print(f"  Wrong signs: {err_summary['wrong_signs']}")
        print(f"  Duplicates: {err_summary['duplicates']}")
        
        print("\n" + "=" * 80)
    
    def _print_confusion_matrix(self, matrix: Dict[str, Dict[str, Any]], is_percent: bool = False):
        """Pretty print confusion matrix"""
        signs = sorted(matrix.keys())
        
        # Header
        header = "GT \\ PRED".ljust(25) + "".join(s.ljust(20) for s in signs)
        print(header)
        print("-" * len(header))
        
        # Rows
        for gt_sign in signs:
            row = gt_sign.ljust(25)
            for pred_sign in signs:
                val = matrix[gt_sign].get(pred_sign, 0)
                if is_percent:
                    row += f"{val}%".ljust(20)
                else:
                    row += str(val).ljust(20)
            print(row)


# ---------- Core evaluation ----------
def evaluate_llm_extraction(
    gt: Dict[str, Dict[str, Dict[str, List[str]]]],
    pred: Dict[str, Dict[str, Dict[str, List[str]]]],
    signs: List[str] = SIGNS,
) -> EvaluationReport:
    """
    Comprehensive evaluation of drug extraction using SUMMATION method.
    All metrics work by summing quantities per (case, drug, sign) and comparing totals.
    
    Args:
        gt: Ground truth data - gt[case_id][drug][sign] = ["X-יחידה", ...]
        pred: Predicted data with same structure
        signs: List of sign types to evaluate
    
    Returns:
        EvaluationReport with all metrics and error analysis
    """
    
    # --- Coverage Analysis ---
    gt_cases = set(gt.keys())
    pred_cases = set(pred.keys())
    
    gt_drugs = set()
    pred_drugs = set()
    for case_id, drugs in gt.items():
        for drug in drugs.keys():
            gt_drugs.add((case_id, drug))
    for case_id, drugs in pred.items():
        for drug in drugs.keys():
            pred_drugs.add((case_id, drug))
    
    coverage = CoverageMetrics(
        gt_cases=gt_cases,
        pred_cases=pred_cases,
        missing_cases=gt_cases - pred_cases,
        extra_cases=pred_cases - gt_cases,
        gt_drugs=gt_drugs,
        pred_drugs=pred_drugs,
        missing_drugs=gt_drugs - pred_drugs,
        extra_drugs=pred_drugs - gt_drugs
    )
    
    # --- Error Analysis Setup ---
    error_analysis = ErrorAnalysis()
    
    # ============================================================
    # SUMMATION-BASED EVALUATION
    # ============================================================
    # Build mapping: (case, drug, sign) -> sum of numeric values
    def build_summed_map(case_to_drugs: Dict) -> Dict[Tuple[str, str, str], float]:
        result = {}
        for case_id, drugs_obj in (case_to_drugs or {}).items():
            if not isinstance(drugs_obj, dict):
                continue
            for drug, sign_obj in drugs_obj.items():
                if not isinstance(sign_obj, dict):
                    continue
                for sign_name in signs:
                    qty_list = []
                    if sign_name in sign_obj:
                        raw = sign_obj[sign_name]
                        if raw is None:
                            qty_list = []
                        elif isinstance(raw, (list, tuple)):
                            qty_list = [str(v).strip() for v in raw if str(v).strip()]
                        elif isinstance(raw, str):
                            s = raw.strip()
                            qty_list = [s] if s else []
                        else:
                            s = str(raw).strip()
                            qty_list = [s] if s else []
                    
                    total = 0.0
                    for qty_str in qty_list:
                        val = _extract_numeric_value(qty_str)
                        if val is not None:
                            total += val
                    if total > 0:  # Only include non-zero sums
                        result[(case_id, drug, sign_name)] = total
        return result
    
    gt_summed = build_summed_map(gt)
    pr_summed = build_summed_map(pred)
    
    # STRICT matching with tolerance for floating point
    EPSILON = 0.0001
    tp = 0
    fp = 0
    fn = 0
    
    matched_keys = set()
    
    # Find matches
    for key in gt_summed.keys():
        if key in pr_summed and abs(gt_summed[key] - pr_summed[key]) < EPSILON:
            tp += 1
            matched_keys.add(key)
        else:
            fn += 1
            case_id, drug, sign = key
            error_analysis.false_negatives.append(
                ErrorExample(case_id, drug, f"{gt_summed[key]:.2f}", sign, None, 'FN')
            )
    
    for key in pr_summed.keys():
        if key not in matched_keys:
            fp += 1
            case_id, drug, sign = key
            error_analysis.false_positives.append(
                ErrorExample(case_id, drug, f"{pr_summed[key]:.2f}", None, sign, 'FP')
            )
    
    strict_precision = _safe_div(tp, tp + fp)
    strict_recall = _safe_div(tp, tp + fn)
    strict_f1 = _f1(strict_precision, strict_recall)
    
    strict = StrictMetrics(
        tp=tp, fp=fp, fn=fn,
        precision=strict_precision,
        recall=strict_recall,
        f1=strict_f1
    )
    
    # QUANTITY-ONLY (ignoring sign, just case+drug)
    def build_summed_no_sign(case_to_drugs: Dict) -> Dict[Tuple[str, str], float]:
        result = defaultdict(float)
        for case_id, drugs_obj in (case_to_drugs or {}).items():
            if not isinstance(drugs_obj, dict):
                continue
            for drug, sign_obj in drugs_obj.items():
                if not isinstance(sign_obj, dict):
                    continue
                for sign_name in signs:
                    qty_list = []
                    if sign_name in sign_obj:
                        raw = sign_obj[sign_name]
                        if raw is None:
                            qty_list = []
                        elif isinstance(raw, (list, tuple)):
                            qty_list = [str(v).strip() for v in raw if str(v).strip()]
                        elif isinstance(raw, str):
                            s = raw.strip()
                            qty_list = [s] if s else []
                        else:
                            s = str(raw).strip()
                            qty_list = [s] if s else []
                    
                    for qty_str in qty_list:
                        val = _extract_numeric_value(qty_str)
                        if val is not None:
                            result[(case_id, drug)] += val
        return dict(result)
    
    gt_summed_no_sign = build_summed_no_sign(gt)
    pr_summed_no_sign = build_summed_no_sign(pred)
    
    tp_q = 0
    fp_q = 0
    fn_q = 0
    
    for key in gt_summed_no_sign.keys():
        if key in pr_summed_no_sign and abs(gt_summed_no_sign[key] - pr_summed_no_sign[key]) < EPSILON:
            tp_q += 1
        else:
            fn_q += 1
    
    for key in pr_summed_no_sign.keys():
        if key not in gt_summed_no_sign or abs(gt_summed_no_sign.get(key, 0) - pr_summed_no_sign[key]) >= EPSILON:
            fp_q += 1
    
    q_precision = _safe_div(tp_q, tp_q + fp_q)
    q_recall = _safe_div(tp_q, tp_q + fn_q)
    q_f1 = _f1(q_precision, q_recall)
    
    quantity_only = QuantityOnlyMetrics(
        tp=tp_q, fp=fp_q, fn=fn_q,
        precision=q_precision,
        recall=q_recall,
        f1=q_f1
    )
    
    # SIGN classification (for matched case+drug pairs)
    confusion: Dict[str, Dict[str, int]] = {gs: {ps: 0 for ps in signs} for gs in signs}
    correct = 0
    total_matched = 0
    
    # For each (case, drug) that exists in both
    matched_case_drugs = set()
    for key in set(gt_summed_no_sign.keys()) & set(pr_summed_no_sign.keys()):
        if abs(gt_summed_no_sign[key] - pr_summed_no_sign[key]) < EPSILON:
            matched_case_drugs.add(key)
    
    for case_drug in matched_case_drugs:
        case_id, drug = case_drug
        
        # Get sign distribution for this case+drug
        gt_sign_dist = {}
        pr_sign_dist = {}
        
        for sign in signs:
            gt_key = (case_id, drug, sign)
            pr_key = (case_id, drug, sign)
            gt_sign_dist[sign] = gt_summed.get(gt_key, 0.0)
            pr_sign_dist[sign] = pr_summed.get(pr_key, 0.0)
        
        # Check which signs match
        for sign in signs:
            if abs(gt_sign_dist[sign] - pr_sign_dist[sign]) < EPSILON and gt_sign_dist[sign] > 0:
                correct += 1
                total_matched += 1
                confusion[sign][sign] += 1
            elif gt_sign_dist[sign] > 0 or pr_sign_dist[sign] > 0:
                total_matched += 1
                # Find where the quantity went wrong
                if gt_sign_dist[sign] > 0 and pr_sign_dist[sign] == 0:
                    # GT has this sign, PRED doesn't - find where PRED put it
                    for other_sign in signs:
                        if other_sign != sign and pr_sign_dist[other_sign] > 0:
                            confusion[sign][other_sign] += 1
                            error_analysis.wrong_signs.append(
                                ErrorExample(case_id, drug, f"{gt_sign_dist[sign]:.2f}", sign, other_sign, 'WRONG_SIGN')
                            )
                            break
    
    sign_acc = _safe_div(correct, total_matched)
    sign_metrics = SignMetrics(
        correct=correct,
        total_matched_items=total_matched,
        accuracy=sign_acc,
        confusion=confusion
    )
    
    # --- Duplicate rates (both gt and pred) ---
    def duplicate_metrics(case_to_drugs: Dict, error_list: List[ErrorExample] = None) -> DuplicateMetrics:
        total_groups = 0
        groups_with_dups = 0
        duplicate_items = 0
        
        for case_id, drugs_obj in (case_to_drugs or {}).items():
            if not isinstance(drugs_obj, dict):
                continue
            for drug, sign_obj in drugs_obj.items():
                if not isinstance(sign_obj, dict):
                    continue
                for sign_name in signs:
                    qty_list = []
                    if sign_name in sign_obj:
                        raw = sign_obj[sign_name]
                        if raw is None:
                            qty_list = []
                        elif isinstance(raw, (list, tuple)):
                            qty_list = [str(v).strip() for v in raw if str(v).strip()]
                        elif isinstance(raw, str):
                            s = raw.strip()
                            qty_list = [s] if s else []
                        else:
                            s = str(raw).strip()
                            qty_list = [s] if s else []
                    
                    if not qty_list:
                        continue
                    total_groups += 1
                    c = Counter(qty_list)
                    extras = sum(v - 1 for v in c.values() if v > 1)
                    if extras > 0:
                        groups_with_dups += 1
                        duplicate_items += extras
                        # Record duplicate examples
                        if error_list is not None:
                            for qty, count in c.items():
                                if count > 1:
                                    for _ in range(count - 1):
                                        error_list.append(
                                            ErrorExample(case_id, drug, qty, sign_name, sign_name, 'DUPLICATE')
                                        )
        
        dup_rate_groups = _safe_div(groups_with_dups, total_groups)
        return DuplicateMetrics(
            total_groups=total_groups,
            groups_with_dups=groups_with_dups,
            duplicate_items=duplicate_items,
            duplicate_rate_groups=dup_rate_groups,
        )
    
    dup_pred = duplicate_metrics(pred, error_analysis.duplicates)
    dup_gt = duplicate_metrics(gt)

    # --- Specificity Analysis (Empty Cell Accuracy) ---
    # Count all (case, drug, sign) cells
    def get_all_cells(case_to_drugs: Dict, all_cases: Set[str], all_drugs: Set[str]) -> Set[Tuple[str, str, str]]:
        """Get all possible (case, drug, sign) cells"""
        all_possible_cells = set()
        for case_id in (gt_cases | pred_cases):
            gt_drugs_case = set(gt.get(case_id, {}).keys()) if isinstance(gt.get(case_id, {}), dict) else set()
            pr_drugs_case = set(pred.get(case_id, {}).keys()) if isinstance(pred.get(case_id, {}), dict) else set()
            for drug in (gt_drugs_case | pr_drugs_case):
                for sign in signs:
                    all_possible_cells.add((case_id, drug, sign))
        return all_possible_cells
            
    # Get all unique drugs from both GT and PRED
    all_drugs = set()
    for case_id, drugs in gt.items():
        all_drugs.update(drugs.keys())
    for case_id, drugs in pred.items():
        all_drugs.update(drugs.keys())
    
    all_cases = gt_cases | pred_cases
    all_possible_cells = get_all_cells(gt, all_cases, all_drugs)
    
    # Identify which cells are empty in GT
    gt_empty_cells = set()
    gt_filled_cells = set()
    for case_id, drugs_obj in gt.items():
        if not isinstance(drugs_obj, dict):
            continue
        for drug, sign_obj in drugs_obj.items():
            if not isinstance(sign_obj, dict):
                continue
            for sign in signs:
                qty_list = _as_list(sign_obj.get(sign, []))
                cell = (case_id, drug, sign)
                if not qty_list:
                    gt_empty_cells.add(cell)
                else:
                    gt_filled_cells.add(cell)
    
    # For cells not in GT at all, consider them empty
    for cell in all_possible_cells:
        case_id, drug, sign = cell
        if case_id in gt and drug in gt.get(case_id, {}):
            continue  # Already processed above
        gt_empty_cells.add(cell)
    
    # Identify which cells are empty in PRED
    pred_empty_cells = set()
    pred_filled_cells = set()
    for case_id, drugs_obj in pred.items():
        if not isinstance(drugs_obj, dict):
            continue
        for drug, sign_obj in drugs_obj.items():
            if not isinstance(sign_obj, dict):
                continue
            for sign in signs:
                qty_list = _as_list(sign_obj.get(sign, []))
                cell = (case_id, drug, sign)
                if not qty_list:
                    pred_empty_cells.add(cell)
                else:
                    pred_filled_cells.add(cell)
    
    # For cells not in PRED at all, consider them empty
    for cell in all_possible_cells:
        case_id, drug, sign = cell
        if case_id in pred and drug in pred.get(case_id, {}):
            continue  # Already processed above
        pred_empty_cells.add(cell)
    
    # Calculate specificity: TN / (TN + FP_empty)
    # TN = cells that are empty in both GT and PRED
    # FP_empty = cells that are empty in GT but filled in PRED
    true_negatives = len(gt_empty_cells & pred_empty_cells)
    false_positives_empty = len(gt_empty_cells & pred_filled_cells)
    total_empty_cells = len(gt_empty_cells)
    
    specificity_value = _safe_div(true_negatives, true_negatives + false_positives_empty)
    
    specificity = SpecificityMetrics(
        total_empty_cells=total_empty_cells,
        true_negatives=true_negatives,
        false_positives_empty=false_positives_empty,
        specificity=specificity_value
    )
    
    # --- Quantity Distance Analysis ---
    # Overall: For each (case, drug), extract all numeric values and compare
    def build_case_drug_quantities(case_to_drugs: Dict) -> Dict[Tuple[str, str], List[float]]:
        """Extract all numeric quantities for each (case, drug) pair"""
        result = defaultdict(list)
        for case_id, drugs_obj in (case_to_drugs or {}).items():
            if not isinstance(drugs_obj, dict):
                continue
            for drug, sign_obj in drugs_obj.items():
                if not isinstance(sign_obj, dict):
                    continue
                for sign_name in signs:
                    qty_list = _as_list(sign_obj.get(sign_name, []))
                    for qty_str in qty_list:
                        val = _extract_numeric_value(qty_str)
                        if val is not None:
                            result[(case_id, drug)].append(val)
        return result
    
    # Per-sign: For each (case, drug, sign), extract numeric values
    def build_case_drug_sign_quantities(case_to_drugs: Dict) -> Dict[Tuple[str, str, str], List[float]]:
        """Extract all numeric quantities for each (case, drug, sign) triple"""
        result = defaultdict(list)
        for case_id, drugs_obj in (case_to_drugs or {}).items():
            if not isinstance(drugs_obj, dict):
                continue
            for drug, sign_obj in drugs_obj.items():
                if not isinstance(sign_obj, dict):
                    continue
                for sign_name in signs:
                    qty_list = _as_list(sign_obj.get(sign_name, []))
                    for qty_str in qty_list:
                        val = _extract_numeric_value(qty_str)
                        if val is not None:
                            result[(case_id, drug, sign_name)].append(val)
        return result
    
    gt_qty_map = build_case_drug_quantities(gt)
    pr_qty_map = build_case_drug_quantities(pred)
    
    gt_qty_sign_map = build_case_drug_sign_quantities(gt)
    pr_qty_sign_map = build_case_drug_sign_quantities(pred)
    
    # Calculate overall distances
    all_errors = []
    near_misses = []
    exact_matches = 0
    total_comparisons = 0
    
    for key in set(gt_qty_map.keys()) | set(pr_qty_map.keys()):
        gt_vals = sorted(gt_qty_map.get(key, []))
        pr_vals = sorted(pr_qty_map.get(key, []))
        
        # if not gt_vals or not pr_vals:
        #     continue
        
        total_comparisons += 1
        
        # Compare sum of quantities
        gt_sum = sum(gt_vals)
        pr_sum = sum(pr_vals)
        
        error = abs(gt_sum - pr_sum)
        all_errors.append(error)
        
        if error == 0< EPSILON:
            exact_matches += 1
        elif error > 0:
            case_id, drug = key
            near_misses.append((case_id, drug, gt_sum, pr_sum, error))
    
    # Sort near misses by error (smallest first)
    near_misses.sort(key=lambda x: x[4])
    def _median(xs: List[float]) -> float:
        if not xs:
            return 0.0
        ys = sorted(xs)
        n = len(ys)
        mid = n // 2
        return ys[mid] if n % 2 == 1 else (ys[mid - 1] + ys[mid]) / 2.0

    mean_abs_error = sum(all_errors) / len(all_errors) if all_errors else 0.0
    median_abs_error = _median(all_errors)    

    # Calculate per-sign distances
    per_sign_metrics = {}
    for sign_name in signs:
        sign_errors = []
        sign_near_misses = []
        sign_exact = 0
        sign_comparisons = 0
        
        for key in set(gt_qty_sign_map.keys()) | set(pr_qty_sign_map.keys()):
            case_id, drug, sign = key
            if sign != sign_name:
                continue
                
            gt_vals = sorted(gt_qty_sign_map.get(key, []))
            pr_vals = sorted(pr_qty_sign_map.get(key, []))
            
            if not gt_vals or not pr_vals:
                continue
            
            sign_comparisons += 1
            
            gt_sum = sum(gt_vals)
            pr_sum = sum(pr_vals)
            
            error = abs(gt_sum - pr_sum)
            sign_errors.append(error)
            
            if error == 0:
                sign_exact += 1
            elif error > 0:
                sign_near_misses.append((case_id, drug, gt_sum, pr_sum, error))
        
        sign_near_misses.sort(key=lambda x: x[4])
        
        per_sign_metrics[sign_name] = {
            'comparisons': sign_comparisons,
            'exact_matches': sign_exact,
            'exact_match_rate': _safe_div(sign_exact, sign_comparisons),
            'mean_abs_error': sum(sign_errors) / len(sign_errors) if sign_errors else 0.0,
            'median_abs_error': sorted(sign_errors)[len(sign_errors) // 2] if sign_errors else 0.0,
            'near_misses': sign_near_misses
        }
    
    quantity_distance = QuantityDistanceMetrics(
        total_comparisons=total_comparisons,
        exact_matches=exact_matches,
        near_misses=near_misses,
        mean_absolute_error=mean_abs_error,
        median_absolute_error=median_abs_error,
        per_sign_metrics=per_sign_metrics
    )
    
    return EvaluationReport(
        strict=strict,
        quantity_only=quantity_only,
        sign=sign_metrics,
        duplicates_pred=dup_pred,
        duplicates_gt=dup_gt,
        coverage=coverage,
        error_analysis=error_analysis,
        quantity_distance=quantity_distance,
        specificity=specificity
    )

drug_columns = ['lsd', 'methamphetamine', 'ayahuasca', 'cathinone', 'ketamine',
                'hashish', 'methylmethcathinone', 'cannabis',
                'mdma', 'cocaine', 'heroin']

def clean_sign_a_non_personal(cell):
    """
    cell: dict like {
        'sign_a_non_personal': [...],
        'sign_a_personal': [...],
        'sign_b': [...]
    }
    """
    if pd.isna(cell) or cell == "" or cell == "{}":
        return cell

    if isinstance(cell, str):
        try:
            cell = ast.literal_eval(cell)
        except Exception:
            return cell

    if not isinstance(cell, dict):
        return cell

    a_non = set(cell.get('sign_a_non_personal', []))
    a_personal = set(cell.get('sign_a_personal', []))
    b = set(cell.get('sign_b', []))

    # remove overlaps
    cleaned_a_non = list(a_non - a_personal - b)

    cell['sign_a_non_personal'] = cleaned_a_non
    return cell


LIST_KEYS = ['sign_a_non_personal', 'sign_a_personal', 'sign_b']

def parse_amount(val):
    """
    '55-גרם' -> (55.0, 'גרם')
    """
    m = re.match(r'([\d\.]+)\s*-\s*(.+)', val)
    if not m:
        return None, None
    return float(m.group(1)), m.group(2)

def format_amount(num, unit):
    # שמירה על ניקיון מספרי
    if num.is_integer():
        num = int(num)
    return f"{num}-{unit}"

def split_hashish_cannabis(cell_hashish, cell_cannabis):
    """
    מחלק חצי-חצי ערכים חופפים בין hashish ל-cannabis
    """
    if not isinstance(cell_hashish, dict) or not isinstance(cell_cannabis, dict):
        return cell_hashish, cell_cannabis

    for key in LIST_KEYS:
        h_vals = cell_hashish.get(key, [])
        c_vals = cell_cannabis.get(key, [])

        overlap = set(h_vals) & set(c_vals)

        if not overlap:
            continue

        new_h, new_c = [], []

        for v in h_vals:
            if v in overlap:
                num, unit = parse_amount(v)
                if num is not None:
                    new_h.append(format_amount(num / 2, unit))
                else:
                    new_h.append(v)
            else:
                new_h.append(v)

        for v in c_vals:
            if v in overlap:
                num, unit = parse_amount(v)
                if num is not None:
                    new_c.append(format_amount(num / 2, unit))
                else:
                    new_c.append(v)
            else:
                new_c.append(v)

        cell_hashish[key] = new_h
        cell_cannabis[key] = new_c

    return cell_hashish, cell_cannabis


# ---------- Example usage ----------
if __name__ == "__main__":
    import pandas as pd
    import re

    
    # Uncomment to load from CSV files
    df_gt = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\GT.csv", encoding="utf-8-sig")
    # for drug in drug_columns:
    #     df_gt[drug] = df_gt[drug].apply(clean_sign_a_non_personal)

    gt = {}
    for index, row in df_gt.iterrows():
        case_id = row['original_case']
        gt[case_id] = {}
        for drug in drug_columns:
            gt[case_id][drug] = {}
            drug_data = row[drug]
            # Handle both string and dict cases
            if isinstance(drug_data, str):
                try:
                    drug_data = ast.literal_eval(drug_data)
                except:
                    # If eval fails, create empty dict
                    drug_data = {sign: [] for sign in SIGNS}
            elif not isinstance(drug_data, dict):
                # If it's neither string nor dict, create empty dict
                drug_data = {sign: [] for sign in SIGNS}
            
            for sign in SIGNS:
                gt[case_id][drug][sign] = _as_list(drug_data.get(sign, []))
    
    df_pred = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\PRED.csv", encoding="utf-8-sig")
    # for drug in drug_columns:
    #     df_pred[drug] = df_pred[drug].apply(clean_sign_a_non_personal)

    for idx in df_pred.index:
        h = df_pred.at[idx, 'hashish']
        c = df_pred.at[idx, 'cannabis']

        # parse if string
        if isinstance(h, str):
            h = ast.literal_eval(h)
        if isinstance(c, str):
            c = ast.literal_eval(c)
#
        # h, c = split_hashish_cannabis(h, c)

        df_pred.at[idx, 'hashish'] = h
        df_pred.at[idx, 'cannabis'] = c
    pred = {}
    for index, row in df_pred.iterrows():
        case_id = row['original_case']
        pred[case_id] = {}
        for drug in drug_columns:
            pred[case_id][drug] = {}
            drug_data = row[drug]
            # Handle both string and dict cases
            if isinstance(drug_data, str):
                try:
                    drug_data = ast.literal_eval(drug_data)
                except:
                    # If eval fails, create empty dict
                    drug_data = {sign: [] for sign in SIGNS}
            elif not isinstance(drug_data, dict):
                # If it's neither string nor dict, create empty dict
                drug_data = {sign: [] for sign in SIGNS}
            
            for sign in SIGNS:
                pred[case_id][drug][sign] = _as_list(drug_data.get(sign, []))
    
    report = evaluate_llm_extraction(gt, pred)
    report.print_summary()

